## === WORKFLOW ===

**Phase 1 (GEE) → `04_export_modis_igbp.ipynb`:**
- Lade MODIS MCD12Q1 IGBP für alle Jahre 2001–2024
- Reproject zu 500m EPSG:3035, Clippe auf wildE-Grid
- Export als 24-Band GeoTIFF → `MODIS_IGBP_2001_2024_500m_wildE.tif`

**Phase 2 (Lokal) → `04_export_modis_igbp.ipynb`:**
- Download via PyDrive + Validierung gegen Grid-Referenz
- *(Kein combined TIF — alle Dateien werden in Phase 3 separat geladen)*

**Phase 3 (Lokal) — diese Datei:**

| Schritt | Beschreibung |
|---|---|
| **0** | Lade alle Daten (Woody, MBA, MODIS IGBP, Ecoregions — separat) |
| **1** | Pre-Fire Landcover (MODIS IGBP Jahr vor Brand) |
| **2a–d** | LC-Trajektorien (Major-Klasse, IGBP-Einzelklasse, LC×FireCount, LC×Ecoregion) |
| **2e** | Eco-Trajektorien (1 Fire) |
| **2f** | Eco × Fire Count |
| **2g** | Fire Statistics (vektorisiert) |
| **2h** | Recovery-%-Index (loss-relativ, Fire Freq 1–4, +10yr) |
| **2i** | Heatmap Ecoregion × LC (Fläche km²/%) |
| **3** | LC-Visualisierungen (Plots 1–5) |
| **3b** | Stat. Tests LC (Kruskal-Wallis + Dunn, europa-weit & pro Ecoregion) |
| **3c** | Eco-Visualisierungen (E1–E10) |
| **3d** | Stat. Tests Ecoregion (Kruskal-Wallis + Dunn) |
| **4** | CSV-Export + Abschlussmeldung |
| **★** | Schnelle Visualisierung aus CSV (eine Zelle, ganz am Ende) |


## Kurzübersicht

| Phase | Wo | Was |
|---|---|---|
| **Phase 1** | GEE | MODIS MCD12Q1 IGBP Export (2001–2024) → 24-Band GeoTIFF |
| **Phase 2** | Lokal | Download + Validierung `MODIS_IGBP_2001_2024_500m_wildE.tif` (kein combined TIF) |
| **Phase 3** | Lokal | Analyse → Plots 1–5 (LC), E1–E10 (Eco), Stat. Tests, CSV-Export |

→ Details siehe Workflow-Zelle oben · Phase 1+2 → `04_export_modis_igbp.ipynb`


## Phase 3: MODIS IGBP Land Cover Analysis


In [2]:
# === CONFIGURATION & SETUP ===
# All imports, paths, constants, and output directories

import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

_workDir_remote = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
_workDir_local  = r"D:\Seafile\Meine Bibliothek\uni\master\thesis"
workDir = _workDir_local
#workDir = _workDir_remote if os.path.exists(_workDir_remote) else _workDir_local
print(f"workDir: {workDir}")

# === TEMPORAL CONFIGURATION ===
years_woody    = list(range(1985, 2025))   # 40 bands (1985-2024)
years_burned   = list(range(2001, 2026))   # 25 years (2001-2025)
MBA_START_YEAR = 2000                      # First year in MBA raster (band index base)
band_structure = [
    "burned", "count", "doy_1", "doy_2", "doy_3", "doy_4",
    "doy_min", "doy_max", "doy_span", "month_first", "month_last", "season_length"
]
modis_years = list(range(2001, 2025))      # 24 bands (2001-2024)
rel_years   = list(range(-5, 11))          # trajectory window: -5 to +10

# === FILE PATHS ===
woody_path  = os.path.join(workDir, r"_Runs\02_Woody_Cover\woody_cover_500m_3035.tif")
burned_path = os.path.join(workDir, r"_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m_mosaic.tif")

ecoregion_raster_path = os.path.join(workDir, r"_Runs\03_Ecoregions_EEA\ecoregions_500m_3035_MBA.tif")
ecoregion_attr_path   = os.path.join(workDir, r"_Runs\03_Ecoregions_EEA\ecoregions_attributes_with_colors.csv")

modis_path = os.path.join(workDir, r"_Runs\04_Landcover\MODIS_GEE_tiles\MODIS_IGBP_2001_2024_500m_wildE.tif")

# === EXCLUSIONS ===
LC_EXCLUDE  = {'Barren/Ice'}   # too rare / irrelevant for Europe
ECO_EXCLUDE = {'OUT'}          # "Outside" = outside EEA study area

# === PIXEL AREA ===
pixel_area_km2 = 0.25   # 500 m × 500 m
pixel_area_ha  = 25.0

# === COLOR DEFINITIONS ===
LC_COLORS = {
    'Forests':       '#228B22',
    'Shrublands':    '#8B4513',
    'Open Woodland': '#DAA520',
    'Grasslands':    '#90EE90',
    'Wetlands':      '#4682B4',
    'Croplands':     '#FFD700',
}

FIRE_COUNT_COLORS = {
    1: '#1f77b4',
    2: '#ff7f0e',
    3: '#2ca02c',
    4: '#d62728',
}

# === OUTPUT DIRECTORIES (09_Results) ===
results_base = os.path.join(workDir, "_Runs", "09_Results")

general_plots_dir = os.path.join(results_base, "General",        "plots")
general_csv_dir   = os.path.join(results_base, "General",        "csv")
eco_plots_dir     = os.path.join(results_base, "Ecoregions",     "plots")
eco_csv_dir       = os.path.join(results_base, "Ecoregions",     "csv")
lc_plots_dir      = os.path.join(results_base, "Landcover",      "plots")
lc_csv_dir        = os.path.join(results_base, "Landcover",      "csv")
lcxeco_plots_dir  = os.path.join(results_base, "LC_x_Ecoregion", "plots")
lcxeco_csv_dir    = os.path.join(results_base, "LC_x_Ecoregion", "csv")

for d in [general_plots_dir, general_csv_dir,
          eco_plots_dir, eco_csv_dir,
          lc_plots_dir, lc_csv_dir,
          lcxeco_plots_dir, lcxeco_csv_dir]:
    os.makedirs(d, exist_ok=True)

print("=" * 70)
print("WOODY COVER TRAJECTORY ANALYSIS")
print("=" * 70)
print(f"  Woody Cover : {years_woody[0]}-{years_woody[-1]}  ({len(years_woody)} bands)")
print(f"  MBA Burned  : {years_burned[0]}-{years_burned[-1]}  ({len(years_burned)} years × {len(band_structure)} metrics)")
print(f"  MODIS IGBP  : {modis_years[0]}-{modis_years[-1]}  ({len(modis_years)} bands)")
print(f"\n  Output:")
print(f"    General:        {general_plots_dir}")
print(f"    Ecoregions:     {eco_plots_dir}")
print(f"    Landcover:      {lc_plots_dir}")
print(f"    LC×Ecoregion:   {lcxeco_plots_dir}")

workDir: D:\Seafile\Meine Bibliothek\uni\master\thesis
WOODY COVER TRAJECTORY ANALYSIS
  Woody Cover : 1985-2024  (40 bands)
  MBA Burned  : 2001-2025  (25 years × 12 metrics)
  MODIS IGBP  : 2001-2024  (24 bands)

  Output:
    General:        D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\09_Results\General\plots
    Ecoregions:     D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\09_Results\Ecoregions\plots
    Landcover:      D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\09_Results\Landcover\plots
    LC×Ecoregion:   D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\09_Results\LC_x_Ecoregion\plots


In [3]:
# ============================================================
# CACHE SETTINGS & LOADER
# ============================================================
import pickle
import colorsys
import matplotlib.colors as mcolors

LOAD_FROM_CACHE = True    # skip computation steps and load results directly
SAVE_CACHE      = True    # save results at the end for future use

cache_dir  = os.path.join(workDir, "_Runs", "09_Results", "_cache")
os.makedirs(cache_dir, exist_ok=True)
cache_file = os.path.join(cache_dir, "calculations.pickle")

SKIP_TO_VISUALIZATION = False

if LOAD_FROM_CACHE and os.path.exists(cache_file):
    print("Loading cached results...")
    try:
        with open(cache_file, 'rb') as f:
            cache_data = pickle.load(f)

        # ── 1. Trajectory result dicts ───────────────────────────────────────
        eco_results           = cache_data.get('eco_results', {})
        eco_firecount_results = cache_data.get('eco_firecount_results', {})
        lc_eco_results        = cache_data.get('lc_eco_results', {})
        lc_major_results      = cache_data.get('lc_major_results', {})
        lc_firecount_results  = cache_data.get('lc_firecount_results', {})
        lc_detail_results     = cache_data.get('lc_detail_results', {})

        print(f"  Trajectory dicts:      eco={len(eco_results)}, eco_fc={len(eco_firecount_results)}, "
              f"lc={len(lc_major_results)}, lc_fc={len(lc_firecount_results)}, lc_eco={len(lc_eco_results)}")

        # ── 2. Fire statistics DataFrames (from Step 2g) ─────────────────────
        burned_area_total  = cache_data.get('burned_area_total',  pd.DataFrame())
        burned_area_eco    = cache_data.get('burned_area_eco',    pd.DataFrame())
        fire_stats_summary = cache_data.get('fire_stats_summary', pd.DataFrame())
        fire_count_eco     = cache_data.get('fire_count_eco',     pd.DataFrame())
        season_stats       = cache_data.get('season_stats',       pd.DataFrame())
        season_by_fc       = cache_data.get('season_by_fc',       {})

        print(f"  Fire stats DFs:        burned_total={len(burned_area_total)}, "
              f"burned_eco={len(burned_area_eco)}, fire_summary={len(fire_stats_summary)}")
        print(f"  season_by_fc:          {sorted(season_by_fc.keys()) if season_by_fc else '(not in cache yet)'}")

        # ── 2b. fire_counts (benötigt von Step 2g) ───────────────────────────
        fire_counts = cache_data.get('fire_counts', None)
        if fire_counts is None:
            print("  fire_counts nicht im Cache — berechne aus Burned-Raster...")
            with rasterio.open(burned_path) as src:
                _burned_bands = [
                    1 + (year - MBA_START_YEAR) * len(band_structure)
                    for year in years_burned
                ]
                _burned_tmp = src.read(_burned_bands)
            fire_counts = np.sum(_burned_tmp == 1, axis=0)
            del _burned_tmp
            print(f"  fire_counts berechnet: {fire_counts.shape}, max={fire_counts.max()}")
        else:
            print(f"  fire_counts aus Cache: {fire_counts.shape}, max={fire_counts.max()}")

        # MBA raster bands — loaded on demand by Eco Vis chunk (E6/E7)
        mba_count  = None
        mba_season = None

        # ── 3. Ecoregion lookup tables (CSV + raster, fast <1s) ──────────────
        ecoregion_df = pd.read_csv(ecoregion_attr_path)
        eco_mapping  = {
            int(row['NUMERIC_ID']): {
                'code':      row['code'],
                'name':      row['name'],
                'hex_color': row['hex_color']
            }
            for _, row in ecoregion_df.iterrows()
        }

        with rasterio.open(ecoregion_raster_path) as src:
            eco_raster = src.read(1)

        unique_eco_ids = np.array([
            eid for eid in np.unique(eco_raster[eco_raster > 0]).astype(int)
            if eco_mapping[eid]['code'] not in ECO_EXCLUDE
        ])

        print(f"  Ecoregion lookups:     eco_mapping={len(eco_mapping)}, "
              f"unique_eco_ids={len(unique_eco_ids)}")

        # ── 4. Helper variables ──────────────────────────────────────────────
        POST_FIRE_YEARS   = list(range(0, 11))
        post_fire_indices = list(range(5, 16))

        recovery_plots_dir = os.path.join(eco_plots_dir, "recovery_normalized")
        os.makedirs(recovery_plots_dir, exist_ok=True)

        # ── 5. Helper functions ──────────────────────────────────────────────
        def _is_excluded(eco_code):
            """Returns True if the ecoregion should be excluded from plots."""
            if eco_code in ECO_EXCLUDE:
                return True
            name = eco_results.get(eco_code, {}).get('name', '')
            return any(ex.lower() in name.lower() or ex.lower() == eco_code.lower()
                       for ex in ECO_EXCLUDE)

        ALL_ECO_CODES = sorted(c for c in eco_results.keys() if not _is_excluded(c))
        n_eco_total   = len(ALL_ECO_CODES)
        n_cols        = 3
        n_rows        = int(np.ceil(n_eco_total / n_cols))

        def get_eco_color(eco_code):
            """Returns the hex color for a given ecoregion code."""
            return eco_results.get(eco_code, {}).get('hex_color', '#808080')

        def get_eco_name(eco_code):
            """Returns the full name for a given ecoregion code."""
            return eco_results.get(eco_code, {}).get('name', eco_code)

        def intensify_color(hex_color, fc_val):
            """Darkens a color by reducing HSV brightness based on fire count."""
            rgb = mcolors.to_rgb(hex_color)
            h, s, v = colorsys.rgb_to_hsv(*rgb)
            v_scale = {1: 1.0, 2: 0.82, 3: 0.65, 4: 0.50}
            return colorsys.hsv_to_rgb(h, s, v * v_scale.get(fc_val, 1.0))

        def plot_recovery_panel(ax, traj, color):
            """Plots a normalized recovery trajectory on ax. Returns False if invalid."""
            fire_year_cover = traj[5]
            if fire_year_cover <= 0 or np.isnan(fire_year_cover):
                return False
            traj_post = traj[post_fire_indices] / fire_year_cover
            ax.plot(POST_FIRE_YEARS, traj_post, marker='o', linewidth=2.5, color=color)
            ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5,
                       alpha=0.7, label='Fire Year (= 1.0)')
            ax.set_xlabel('Years After First Fire', fontsize=9)
            ax.set_ylabel('Norm. Woody Cover', fontsize=9)
            ax.set_xlim(-0.3, 10.3)
            ax.set_ylim(0.5, 1.5)
            ax.set_xticks(POST_FIRE_YEARS)
            ax.legend(fontsize=7, loc='lower right')
            ax.grid(True, alpha=0.3)
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(2)
            return True

        print(f"  ALL_ECO_CODES:         {ALL_ECO_CODES}")
        print(f"\n{'='*70}")
        print(f"CACHE LOADED — skip to visualizations.")
        print(f"{'='*70}")
        SKIP_TO_VISUALIZATION = True

    except Exception as e:
        print(f"Cache error: {e}")
        import traceback; traceback.print_exc()
        print("Falling back to full computation...")
        SKIP_TO_VISUALIZATION = False
else:
    if LOAD_FROM_CACHE:
        print("Cache file not found. Running full computation...")
    SKIP_TO_VISUALIZATION = False

Loading cached results...
  Trajectory dicts:      eco=11, eco_fc=11, lc=6, lc_fc=6, lc_eco=11
  Fire stats DFs:        burned_total=25, burned_eco=275, fire_summary=11
  season_by_fc:          [1, 2, 3, 4]
  fire_counts aus Cache: (9660, 10483), max=22
  Ecoregion lookups:     eco_mapping=11, unique_eco_ids=11
  ALL_ECO_CODES:         ['Alpine', 'Anatolian', 'Arctic', 'Atlantic', 'BlackSea', 'Boreal', 'Continental', 'Mediterranean', 'Pannonian', 'Steppic']

CACHE LOADED — skip to visualizations.


In [11]:
# ============================================================
# CACHE SAVE (run after full computation)
# ============================================================

if SAVE_CACHE:
    print("\n" + "="*70)
    print("SAVING RESULTS TO CACHE")
    print("="*70)

    cache_data = {
        # Trajectory result dicts
        'eco_results':           eco_results,
        'eco_firecount_results': eco_firecount_results,
        'lc_eco_results':        lc_eco_results,
        'lc_major_results':      lc_major_results,
        'lc_firecount_results':  lc_firecount_results,
        'lc_detail_results':     lc_detail_results,
        # Fire statistics DataFrames (Step 2g)
        'burned_area_total':     burned_area_total,
        'burned_area_eco':       burned_area_eco,
        'fire_stats_summary':    fire_stats_summary,
        'fire_count_eco':        fire_count_eco,
        'season_stats':          season_stats,
        'season_by_fc':          {k: {'n': v['n'], 'mean': v['mean'],
                                      'values': v['values']}
                                  for k, v in season_by_fc.items()},
        # fire_counts Raster (benötigt von Step 2g)
        'fire_counts':           fire_counts,
    }

    try:
        with open(cache_file, 'wb') as f:
            pickle.dump(cache_data, f)
        print(f"Cache saved: {cache_file}")
        print(f"\n  Next session:")
        print(f"    1. Set LOAD_FROM_CACHE=True")
        print(f"    2. Run Config + Cache cells")
        print(f"    3. Skip to visualization chunks!")
    except Exception as e:
        print(f"Error saving cache: {e}")


SAVING RESULTS TO CACHE
Cache saved: D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\09_Results\_cache\calculations.pickle

  Next session:
    1. Set LOAD_FROM_CACHE=True
    2. Run Config + Cache cells
    3. Skip to visualization chunks!


In [3]:
# === SCHRITT 0: LADE ALLE DATEN ===

print("\n0. LADE DATEN")
print("-" * 70)

# --- Grid-Eigenschaften aus woody_path (Referenz-Raster) ---
with rasterio.open(woody_path) as src:
    nodata    = src.nodata
    transform = src.transform
    crs       = src.crs
    height    = src.height
    width     = src.width

# --- Lade Woody Cover (alle 40 Jahre, 1985–2024) ---
print("Lade Woody Cover (1985-2024)...")
with rasterio.open(woody_path) as src:
    woody = src.read()   # Shape: (40, H, W)

print(f"  ✓ Woody Cover: {woody.shape}")

# --- Lade Burned Area: "burned"-Band pro Jahr (2001-2025) ---
# Band-Index basiert auf MBA_START_YEAR (2000), da das Raster ab Jahr 2000 vorliegt
print(f"Lade Burned Area ({years_burned[0]}-{years_burned[-1]})...")
with rasterio.open(burned_path) as src:
    burned_band_indices = [
        1 + (year - MBA_START_YEAR) * len(band_structure)
        for year in years_burned
    ]
    burned = src.read(burned_band_indices)   # Shape: (25, H, W)

print(f"  ✓ Burned Area: {burned.shape}")

# --- Lade MODIS IGBP (alle 24 Jahre, 2001–2024) ---
print("Lade MODIS IGBP (2001-2024)...")
with rasterio.open(modis_path) as src:
    modis_igbp = src.read()   # Shape: (24, H, W)

print(f"  ✓ MODIS IGBP: {modis_igbp.shape}")

# --- Lade Ecoregion-Raster & Attribute ---
print("Lade Ecoregion-Raster...")
with rasterio.open(ecoregion_raster_path) as src:
    eco_raster = src.read(1)

ecoregion_df = pd.read_csv(ecoregion_attr_path)

eco_mapping = {}
for _, row in ecoregion_df.iterrows():
    eco_mapping[row['NUMERIC_ID']] = {
        'code':      row['code'],
        'name':      row['name'],
        'hex_color': row['hex_color']
    }

unique_eco_ids = np.array([
    eid for eid in np.unique(eco_raster[eco_raster > 0]).astype(int)
    if eco_mapping[eid]['code'] not in ECO_EXCLUDE
])
print(f"  ✓ Ecoregions: {len(unique_eco_ids)} Regionen (exkl. {ECO_EXCLUDE})")
print(f"\n✓ Alle Daten geladen!")



0. LADE DATEN
----------------------------------------------------------------------
Lade Woody Cover (1985-2024)...
  ✓ Woody Cover: (40, 9660, 10483)
Lade Burned Area (2001-2025)...
  ✓ Burned Area: (25, 9660, 10483)
Lade MODIS IGBP (2001-2024)...
  ✓ MODIS IGBP: (24, 9660, 10483)
Lade Ecoregion-Raster...
  ✓ Ecoregions: 11 Regionen (exkl. {'OUT'})

✓ Alle Daten geladen!


In [4]:
# === SCHRITT 1: BESTIMME PRE-FIRE LANDCOVER PRO PIXEL ===
# Für jedes Pixel das MODIS IGBP Landcover VOR dem ersten Brand verwenden

print("\n1. BESTIMME PRE-FIRE LANDCOVER PRO PIXEL")
print("-" * 70)

# IGBP Klassen-Definition
igbp_classes = {
    1: "Evergreen Needleleaf Forests",
    2: "Evergreen Broadleaf Forests",
    3: "Deciduous Needleleaf Forests",
    4: "Deciduous Broadleaf Forests",
    5: "Mixed Forests",
    6: "Closed Shrublands",
    7: "Open Shrublands",
    8: "Woody Savannas",
    9: "Savannas",
    10: "Grasslands",
    11: "Permanent Wetlands",
    12: "Cropland",
    13: "Urban and Built-up Lands",
    14: "Cropland/Natural Vegetation Mosaics",
    15: "Permanent Snow and Ice",
    16: "Barren",
    17: "Water Bodies"
}

# Lookup-Array für Major-Klassen (Index = IGBP Code, Wert = Major-Klassen-ID)
MAJOR_LC_NAMES = ['', 'Forests', 'Shrublands', 'Open Woodland', 'Grasslands',
                  'Wetlands', 'Croplands', 'Urban', 'Barren/Ice', 'Water', 'Other']
igbp_to_major_id = np.array([
    0,   # 0: NoData
    1,   # 1: Evergreen Needleleaf → Forests
    1,   # 2: Evergreen Broadleaf → Forests
    1,   # 3: Deciduous Needleleaf → Forests
    1,   # 4: Deciduous Broadleaf → Forests
    1,   # 5: Mixed Forests → Forests
    2,   # 6: Closed Shrublands → Shrublands
    2,   # 7: Open Shrublands → Shrublands
    3,   # 8: Woody Savannas → Open Woodland
    3,   # 9: Savannas → Open Woodland
    4,   # 10: Grasslands
    5,   # 11: Wetlands
    6,   # 12: Cropland → Croplands
    7,   # 13: Urban
    6,   # 14: Cropland/Veg Mosaic → Croplands
    8,   # 15: Snow/Ice → Barren/Ice
    8,   # 16: Barren → Barren/Ice
    9,   # 17: Water
], dtype=np.uint8)

# --- Berechne Anzahl Brände pro Pixel ---
fire_counts = np.sum(burned == 1, axis=0)
has_fire = fire_counts > 0

# --- Bestimme erstes Brandjahr-Index pro Pixel (vektorisiert) ---
first_fire_idx = np.argmax(burned == 1, axis=0)  # Index in years_burned

# --- Berechne MODIS-Index für Pre-Fire LC (vektorisiert) ---
print("Berechne Pre-Fire Landcover Zuordnung (vektorisiert)...")

# Brandjahr pro Pixel → MODIS-Ziel-Jahr = Brandjahr - 1
fire_year_values = np.array(years_burned)[first_fire_idx]  # shape: (H, W)
target_years = fire_year_values - 1

# Clippe auf MODIS-Verfügbarkeit [2001, 2024]
modis_idx = np.clip(target_years - modis_years[0], 0, len(modis_years) - 1)

# Extrahiere MODIS LC per Fancy Indexing
rows, cols = np.where(has_fire)
pre_fire_lc = np.zeros((height, width), dtype=np.uint8)
pre_fire_lc[rows, cols] = modis_igbp[modis_idx[rows, cols], rows, cols]

# Filtere ungültige Werte (nur 1-17 behalten)
invalid = (pre_fire_lc < 1) | (pre_fire_lc > 17)
pre_fire_lc[invalid] = 0

# --- Erstelle Major-Klassen-ID Raster (vektorisiert via Lookup) ---
pre_fire_lc_major_id = igbp_to_major_id[pre_fire_lc]  # shape: (H, W), dtype: uint8

# --- Barren/Ice ausschließen (Major ID 8) ---
pre_fire_lc_major_id[pre_fire_lc_major_id == 8] = 0

# Statistik
unique_lc, lc_counts = np.unique(pre_fire_lc[pre_fire_lc > 0], return_counts=True)
print(f"\n✓ Pre-Fire Landcover zugeordnet für {np.sum(pre_fire_lc > 0):,} gebrannte Pixel")
print(f"\nVerteilung der Pre-Fire IGBP Klassen:")
for lc, cnt in zip(unique_lc, lc_counts):
    pct = cnt / lc_counts.sum() * 100
    print(f"  {lc:2d} ({igbp_classes.get(lc, '?'):40s}): {cnt:>8,} Pixel ({pct:.1f}%)")

# Major-Klassen Statistik
unique_major, major_counts = np.unique(pre_fire_lc_major_id[pre_fire_lc_major_id > 0], return_counts=True)
print(f"\nVerteilung der Major Landcover-Klassen (exkl. Barren/Ice):")
for mid, cnt in zip(unique_major, major_counts):
    pct = cnt / major_counts.sum() * 100
    print(f"  {MAJOR_LC_NAMES[mid]:20s}: {cnt:>8,} Pixel ({pct:.1f}%)")



1. BESTIMME PRE-FIRE LANDCOVER PRO PIXEL
----------------------------------------------------------------------
Berechne Pre-Fire Landcover Zuordnung (vektorisiert)...

✓ Pre-Fire Landcover zugeordnet für 4,300,774 gebrannte Pixel

Verteilung der Pre-Fire IGBP Klassen:
   1 (Evergreen Needleleaf Forests            ):   45,746 Pixel (1.1%)
   2 (Evergreen Broadleaf Forests             ):    6,240 Pixel (0.1%)
   3 (Deciduous Needleleaf Forests            ):       93 Pixel (0.0%)
   4 (Deciduous Broadleaf Forests             ):   54,896 Pixel (1.3%)
   5 (Mixed Forests                           ):   74,072 Pixel (1.7%)
   6 (Closed Shrublands                       ):    1,647 Pixel (0.0%)
   7 (Open Shrublands                         ):   14,483 Pixel (0.3%)
   8 (Woody Savannas                          ):  202,348 Pixel (4.7%)
   9 (Savannas                                ):  246,406 Pixel (5.7%)
  10 (Grasslands                              ): 1,064,163 Pixel (24.7%)
  11 (Permanent W

In [5]:
# === HILFSFUNKTIONEN: TRAJEKTORIE-BERECHNUNG ===
# Wird von Eco- UND LC-Analyse genutzt → muss vor beiden stehen

def calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, 
                    rel_years=range(-5, 11)):
    """
    Berechnet Pre-/Post-Fire Woody Cover Trajektorie für maskierte Pixel.
    Verwendet NUR Pixel mit GENAU 1 Brandereignis.
    """
    single_fire_mask = (fire_counts == 1) & mask
    n_pixels = np.sum(single_fire_mask)
    if n_pixels < 30:
        return None

    offset = years_burned[0] - years_woody[0]  # 2001 - 1985 = 16

    y_all, x_all = np.where(single_fire_mask)
    fire_idx_subset = first_fire_idx[y_all, x_all]

    trajectory, trajectory_std, trajectory_n = [], [], []

    for rel_year in rel_years:
        woody_band_idx = fire_idx_subset + rel_year + offset - 1
        valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))

        if np.sum(valid) == 0:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            trajectory_n.append(0)
            continue

        values = woody[woody_band_idx[valid], y_all[valid], x_all[valid]].astype(float)
        values = values[(values != nodata) & (~np.isnan(values))]

        if len(values) > 0:
            trajectory.append(np.nanmean(values))
            trajectory_std.append(np.nanstd(values))
            trajectory_n.append(len(values))
        else:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            trajectory_n.append(0)

    return {'trajectory': trajectory, 'std': trajectory_std,
            'n_per_year': trajectory_n, 'n_pixels': int(n_pixels)}


def calc_trajectory_by_fire_count(woody, burned, mask, years_woody, years_burned,
                                   nodata, fire_count_value, rel_years=range(-5, 11)):
    """
    Berechnet Trajektorie für Pixel mit bestimmter Brandanzahl.
    """
    specific_mask = (fire_counts == fire_count_value) & mask
    n_pixels = np.sum(specific_mask)
    if n_pixels < 30:
        return None

    offset = years_burned[0] - years_woody[0]

    y_all, x_all = np.where(specific_mask)
    fire_idx_subset = first_fire_idx[y_all, x_all]

    trajectory, trajectory_std = [], []

    for rel_year in rel_years:
        woody_band_idx = fire_idx_subset + rel_year + offset - 1
        valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))

        if np.sum(valid) == 0:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            continue

        values = woody[woody_band_idx[valid], y_all[valid], x_all[valid]].astype(float)
        values = values[(values != nodata) & (~np.isnan(values))]

        if len(values) > 0:
            trajectory.append(np.nanmean(values))
            trajectory_std.append(np.nanstd(values))
        else:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)

    return {'trajectory': trajectory, 'std': trajectory_std, 'n_pixels': int(n_pixels)}


rel_years = list(range(-5, 11))  # -5 bis +10 Jahre relativ zum Brand

# Major-Klassen für spätere LC-Analyse vorbereiten
SKIP_MAJOR_IDS = {0, 7, 9, 10}  # NoData, Urban, Water, Other
major_classes_present = sorted(set(
    int(mid) for mid in unique_major if mid not in SKIP_MAJOR_IDS
))
print(f"✓ Trajektorie-Funktionen geladen. {len(major_classes_present)} Major LC Klassen vorhanden: "
      f"{[MAJOR_LC_NAMES[m] for m in major_classes_present]}")


✓ Trajektorie-Funktionen geladen. 6 Major LC Klassen vorhanden: ['Forests', 'Shrublands', 'Open Woodland', 'Grasslands', 'Wetlands', 'Croplands']


In [6]:
# === SCHRITT 2: BERECHNE TRAJEKTORIEN NACH LANDCOVER ===

print("\n2. BERECHNE TRAJEKTORIEN NACH LANDCOVER")
print("-" * 70)
print(f"Analysiere {len(major_classes_present)} Major LC Klassen: "
      f"{[MAJOR_LC_NAMES[m] for m in major_classes_present]}")

# === 2a: Trajektorien nach IGBP Major-Klasse (1 Fire) ===
print("\n2a. Trajektorien nach IGBP Major-Klasse (1 Fire Event)...")

lc_major_results = {}

for major_id in tqdm(major_classes_present, desc="Major LC Klassen"):
    lc_name = MAJOR_LC_NAMES[major_id]
    mask = (pre_fire_lc_major_id == major_id)
    result = calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, rel_years)
    if result is not None:
        lc_major_results[lc_name] = result
        print(f"  ✓ {lc_name}: {result['n_pixels']:,} Pixel")
    else:
        print(f"  ✗ {lc_name}: zu wenige Pixel (<30)")

# === 2b: Trajektorien nach IGBP Einzelklasse (1 Fire) ===
print("\n2b. Trajektorien nach IGBP Einzelklasse (1 Fire Event)...")

lc_detail_results = {}

for lc_code in tqdm(unique_lc, desc="IGBP Klassen"):
    if lc_code in [13, 15, 16, 17]:  # Skip Urban, Ice, Barren, Water
        continue
    mask = (pre_fire_lc == lc_code)
    result = calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, rel_years)
    if result is not None:
        lc_detail_results[int(lc_code)] = result
        print(f"  ✓ {lc_code} ({igbp_classes[lc_code]}): {result['n_pixels']:,} Pixel")

# === 2c: Trajektorien nach Landcover × Fire Count ===
print("\n2c. Trajektorien nach Landcover × Fire Count...")

lc_firecount_results = {}

for major_id in tqdm(major_classes_present, desc="LC × Fire Count"):
    lc_name = MAJOR_LC_NAMES[major_id]
    mask = (pre_fire_lc_major_id == major_id)
    lc_firecount_results[lc_name] = {}
    for fc in range(1, 5):
        result = calc_trajectory_by_fire_count(
            woody, burned, mask, years_woody, years_burned, nodata, fc, rel_years
        )
        if result is not None:
            lc_firecount_results[lc_name][fc] = result
            print(f"  ✓ {lc_name} × {fc} Fire(s): {result['n_pixels']:,} Pixel")

# === 2d: Trajektorien nach Landcover × Ecoregion (1 Fire) ===
print("\n2d. Trajektorien nach Landcover × Ecoregion (1 Fire)...")

lc_eco_results = {}

for eco_id in tqdm(unique_eco_ids, desc="LC × Ecoregion"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    lc_eco_results[eco_code] = {}
    for major_id in major_classes_present:
        lc_name = MAJOR_LC_NAMES[major_id]
        mask = (pre_fire_lc_major_id == major_id) & eco_mask
        result = calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, rel_years)
        if result is not None:
            lc_eco_results[eco_code][lc_name] = result

print(f"\n✓ Alle LC-Trajektorien berechnet!")



2. BERECHNE TRAJEKTORIEN NACH LANDCOVER
----------------------------------------------------------------------
Analysiere 6 Major LC Klassen: ['Forests', 'Shrublands', 'Open Woodland', 'Grasslands', 'Wetlands', 'Croplands']

2a. Trajektorien nach IGBP Major-Klasse (1 Fire Event)...


Major LC Klassen:  17%|█▋        | 1/6 [00:11<00:55, 11.05s/it]

  ✓ Forests: 157,654 Pixel


Major LC Klassen:  33%|███▎      | 2/6 [00:12<00:20,  5.16s/it]

  ✓ Shrublands: 12,943 Pixel


...

  ✓ Wetlands × 4 Fire(s): 998 Pixel
  ✓ Croplands × 1 Fire(s): 1,466,288 Pixel
  ✓ Croplands × 2 Fire(s): 570,652 Pixel
  ✓ Croplands × 3 Fire(s): 251,808 Pixel


LC × Fire Count: 100%|██████████| 6/6 [00:16<00:00,  2.70s/it]


  ✓ Croplands × 4 Fire(s): 116,801 Pixel

2d. Trajektorien nach Landcover × Ecoregion (1 Fire)...


LC × Ecoregion: 100%|██████████| 11/11 [00:30<00:00,  2.73s/it]


✓ Alle LC-Trajektorien berechnet!


In [7]:
# === SCHRITT 2e: TRAJEKTORIEN PRO ECOREGION (1 Fire) ===

print("\n2e. Trajektorien pro Ecoregion (1 Fire Event)...")

eco_results = {}

for eco_id in tqdm(unique_eco_ids, desc="Ecoregions (1 Fire)"):
    eco_code = eco_mapping[eco_id]['code']
    eco_name = eco_mapping[eco_id]['name']
    eco_color = eco_mapping[eco_id]['hex_color']
    eco_mask = (eco_raster == eco_id)
    
    result = calc_trajectory(
        woody, burned, eco_mask, years_woody, years_burned, nodata, rel_years
    )
    
    if result is not None:
        eco_results[eco_code] = {
            **result,
            'name': eco_name,
            'hex_color': eco_color,
            'eco_id': int(eco_id)
        }
        print(f"  ✓ {eco_code} ({eco_name}): {result['n_pixels']:,} Pixel")
    else:
        print(f"  ✗ {eco_code} ({eco_name}): zu wenige Pixel (<30)")

print(f"\n✓ {len(eco_results)} Ecoregion-Trajektorien berechnet")


2e. Trajektorien pro Ecoregion (1 Fire Event)...


Ecoregions (1 Fire):   9%|▉         | 1/11 [00:00<00:04,  2.10it/s]

  ✓ Anatolian (Anatolian Bio-geographical Region): 74,752 Pixel


Ecoregions (1 Fire):  18%|█▊        | 2/11 [00:00<00:03,  2.34it/s]

  ✓ BlackSea (Black Sea Bio-geographical Region): 8,331 Pixel


...

  ✓ Atlantic (Atlantic Bio-geographical Region): 37,725 Pixel


Ecoregions (1 Fire):  91%|█████████ | 10/11 [00:07<00:00,  1.65it/s]

  ✓ Outside (outside Europe): 165,297 Pixel


Ecoregions (1 Fire): 100%|██████████| 11/11 [00:07<00:00,  1.47it/s]

  ✓ Arctic (Arctic Bio-geographical Region): 915 Pixel

✓ 11 Ecoregion-Trajektorien berechnet


In [8]:
# === SCHRITT 2f: TRAJEKTORIEN PRO ECOREGION × FIRE COUNT ===

print("\n2f. Trajektorien pro Ecoregion × Fire Count...")

eco_firecount_results = {}

for eco_id in tqdm(unique_eco_ids, desc="Ecoregion × Fire Count"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    
    eco_firecount_results[eco_code] = {}
    
    for fc in range(1, 5):
        result = calc_trajectory_by_fire_count(
            woody, burned, eco_mask, years_woody, years_burned, nodata, fc, rel_years
        )
        if result is not None:
            eco_firecount_results[eco_code][fc] = result
            print(f"  ✓ {eco_code} × {fc} Fire(s): {result['n_pixels']:,} Pixel")

print(f"\n✓ Ecoregion × Fire Count Trajektorien berechnet")


2f. Trajektorien pro Ecoregion × Fire Count...


Ecoregion × Fire Count:   0%|          | 0/11 [00:00<?, ?it/s]

  ✓ Anatolian × 1 Fire(s): 74,752 Pixel
  ✓ Anatolian × 2 Fire(s): 27,558 Pixel
  ✓ Anatolian × 3 Fire(s): 14,773 Pixel


Ecoregion × Fire Count:   9%|▉         | 1/11 [00:01<00:16,  1.62s/it]

  ✓ Anatolian × 4 Fire(s): 9,543 Pixel
  ✓ BlackSea × 1 Fire(s): 8,331 Pixel
  ✓ BlackSea × 2 Fire(s): 2,991 Pixel
  ✓ BlackSea × 3 Fire(s): 1,651 Pixel


...

  ✓ Atlantic × 4 Fire(s): 454 Pixel
  ✓ Outside × 1 Fire(s): 165,297 Pixel
  ✓ Outside × 2 Fire(s): 62,643 Pixel
  ✓ Outside × 3 Fire(s): 37,404 Pixel


Ecoregion × Fire Count:  91%|█████████ | 10/11 [00:19<00:01,  1.78s/it]

  ✓ Outside × 4 Fire(s): 24,587 Pixel
  ✓ Arctic × 1 Fire(s): 915 Pixel


Ecoregion × Fire Count: 100%|██████████| 11/11 [00:20<00:00,  1.87s/it]


✓ Ecoregion × Fire Count Trajektorien berechnet


In [9]:
# === SCHRITT 2g: FIRE STATISTICS (vektorisiert) ===
# Fire Season Length, Fire Count, Burned Area – alles ohne Python-Loops über Pixel

print("\n2g. FIRE STATISTICS (vektorisiert)")
print("-" * 70)

import time
stats_start = time.time()

# === Lade MBA season_length und count Bänder ===
print("Lade MBA Fire Count & Season Length Bänder...")

with rasterio.open(burned_path) as src:
    # "count" = Index 1 in band_structure, "season_length" = Index 11
    count_band_indices  = [1 + (year - MBA_START_YEAR) * len(band_structure) + 1  for year in years_burned]
    season_band_indices = [1 + (year - MBA_START_YEAR) * len(band_structure) + 11 for year in years_burned]

    mba_count  = src.read(count_band_indices)     # (n_years, H, W)
    mba_season = src.read(season_band_indices)    # (n_years, H, W)

print(f"  ✓ MBA Count: {mba_count.shape}")
print(f"  ✓ MBA Season Length: {mba_season.shape}")

# === Pixelfläche ===
pixel_area_km2 = 0.25   # 500m × 500m
pixel_area_ha  = 25.0

# =========================================================
# BURNED AREA TOTAL – DataFrame ['Year', 'Burned_km2']
# → Plots E4, CSV 7
# =========================================================
print("\nBerechne Burned Area (gesamt)...")

rows_total = []
for yr_idx, year in enumerate(years_burned):
    n_burned = int(np.sum(mba_count[yr_idx] > 0))
    rows_total.append({'Year': year, 'Burned_km2': n_burned * pixel_area_km2})

burned_area_total = pd.DataFrame(rows_total)
print(f"  ✓ burned_area_total: {len(burned_area_total)} Jahre | "
      f"Ø {burned_area_total['Burned_km2'].mean():,.0f} km²/yr")

# =========================================================
# BURNED AREA PRO ECOREGION
# → DataFrame ['Year', 'Ecoregion', 'Burned_km2', 'Burned_Fraction']
# → Plots E5, CSV 8
# =========================================================
print("Berechne Burned Area pro Ecoregion...")

rows_eco = []
for yr_idx, year in enumerate(years_burned):
    burned_mask = (mba_count[yr_idx] > 0)
    for eco_id in unique_eco_ids:
        eco_code     = eco_mapping[eco_id]['code']
        eco_mask     = (eco_raster == eco_id)
        eco_total_px = int(np.sum(eco_mask))
        eco_burned_px = int(np.sum(burned_mask & eco_mask))
        rows_eco.append({
            'Year':            year,
            'Ecoregion':       eco_code,
            'Burned_km2':      eco_burned_px * pixel_area_km2,
            'Burned_Fraction': eco_burned_px / eco_total_px if eco_total_px > 0 else 0.0,
        })

burned_area_eco = pd.DataFrame(rows_eco)
print(f"  ✓ burned_area_eco: {len(burned_area_eco)} Eco×Jahr Einträge")

# =========================================================
# FIRE STATS SUMMARY PRO ECOREGION
# → DataFrame ['Ecoregion', 'Season_Length_*', 'Fire_Count_*']
# → Plots E6a, E7a, CSV 11
# =========================================================
print("Berechne Fire Stats Summary pro Ecoregion...")

stats_rows = []
for eco_id in unique_eco_ids:
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)

    s_vals = mba_season[:, eco_mask].flatten().astype(float)
    c_vals = mba_count[:, eco_mask].flatten().astype(float)
    s_vals = s_vals[(s_vals > 0) & np.isfinite(s_vals)]
    c_vals = c_vals[(c_vals > 0) & np.isfinite(c_vals)]

    if len(s_vals) > 0 and len(c_vals) > 0:
        stats_rows.append({
            'Ecoregion':          eco_code,
            'Season_Length_Min':  float(s_vals.min()),
            'Season_Length_Max':  float(s_vals.max()),
            'Season_Length_Mean': float(s_vals.mean()),
            'Season_Length_Std':  float(s_vals.std()),
            'Fire_Count_Min':     float(c_vals.min()),
            'Fire_Count_Max':     float(c_vals.max()),
            'Fire_Count_Mean':    float(c_vals.mean()),
            'Fire_Count_Std':     float(c_vals.std()),
        })

fire_stats_summary = pd.DataFrame(stats_rows)
# Alias für CSV 10 (fire_season_length_by_ecoregion.csv)
season_stats = fire_stats_summary.copy()

print(f"  ✓ fire_stats_summary: {len(fire_stats_summary)} Ecoregionen")
print(f"  {fire_stats_summary[['Ecoregion','Season_Length_Mean','Fire_Count_Mean']].to_string(index=False)}")

# =========================================================
# SEASON LENGTH × FIRE COUNT – dict für Plot E8
# =========================================================
print("\nBerechne Season Length × Fire Count...")

count_flat  = mba_count.flatten()     # uint16 behalten – kein float64-Blowup
season_flat = mba_season.flatten()    # uint16 behalten
valid_mask  = (count_flat > 0) & (season_flat > 0)
count_valid  = count_flat[valid_mask].astype(np.int32)
season_valid = season_flat[valid_mask].astype(np.float32)  # erst jetzt, nach Filter

season_by_fc = {}
for fc_val in np.unique(count_valid):
    fc_mask = (count_valid == fc_val)
    vals = season_valid[fc_mask]
    season_by_fc[int(fc_val)] = {
        'values': vals,
        'n':      int(len(vals)),
        'mean':   float(vals.mean()),
    }

print(f"  ✓ season_by_fc: Fire-Count-Werte = {sorted(season_by_fc.keys())}")
for k, v in sorted(season_by_fc.items()):
    print(f"    {k} Fire(s): n={v['n']:,}, mean season = {v['mean']:.2f} Monate")

# =========================================================
# FIRE COUNT VERTEILUNG PRO ECOREGION
# → DataFrame ['Ecoregion', 'Fire_Count', 'N_Pixels', 'Area_km2']
# → CSV 9
# =========================================================
print("\nBerechne Fire Count Verteilung pro Ecoregion...")

fc_rows = []
for eco_id in unique_eco_ids:
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    for fc_val in range(0, int(fire_counts.max()) + 1):
        n_px = int(np.sum((fire_counts == fc_val) & eco_mask))
        if n_px > 0:
            fc_rows.append({
                'Ecoregion':  eco_code,
                'Fire_Count': fc_val,
                'N_Pixels':   n_px,
                'Area_km2':   n_px * pixel_area_km2,
            })

fire_count_eco = pd.DataFrame(fc_rows)
print(f"  ✓ fire_count_eco: {len(fire_count_eco)} Zeilen")

# === ABSCHLUSS ===
stats_elapsed = time.time() - stats_start
print(f"\n✓ Schritt 2g abgeschlossen ({stats_elapsed:.1f}s)")
print(f"  burned_area_total:  {len(burned_area_total)} Jahre")
print(f"  burned_area_eco:    {len(burned_area_eco)} Eco×Jahr Einträge")
print(f"  fire_stats_summary: {len(fire_stats_summary)} Ecoregionen")
print(f"  season_by_fc:       {sorted(season_by_fc.keys())}")
print(f"  fire_count_eco:     {len(fire_count_eco)} Zeilen")



2g. FIRE STATISTICS (vektorisiert)
----------------------------------------------------------------------
Lade MBA Fire Count & Season Length Bänder...
  ✓ MBA Count: (25, 9660, 10483)
  ✓ MBA Season Length: (25, 9660, 10483)

Berechne Burned Area (gesamt)...
  ✓ burned_area_total: 25 Jahre | Ø 77,784 km²/yr
Berechne Burned Area pro Ecoregion...
  ✓ burned_area_eco: 275 Eco×Jahr Einträge
Berechne Fire Stats Summary pro Ecoregion...
  ✓ fire_stats_summary: 11 Ecoregionen
      Ecoregion  Season_Length_Mean  Fire_Count_Mean
    Anatolian            1.224774         1.048305
     BlackSea            1.025747         1.011661
      Steppic            1.041079         1.015806
  Continental            1.041948         1.014640
       Alpine            1.030403         1.009731
       Boreal            1.008240         1.006211
Mediterranean            1.022932         1.006066
    Pannonian            1.009350         1.004947
     Atlantic            1.010919         1.005570
      Outsid

In [10]:
# === SCHRITT 2h: HELPER & KONSTANTEN FÜR RECOVERY-ANALYSE ===
# (Plots werden in Sektion 4.2c erzeugt)

import colorsys
import matplotlib.colors as mcolors

print("\n2h. RECOVERY-ANALYSE: Definitionen")
print("-" * 70)

POST_FIRE_YEARS   = list(range(0, 11))
post_fire_indices = list(range(5, 16))

recovery_plots_dir = os.path.join(eco_plots_dir, "recovery_normalized")
os.makedirs(recovery_plots_dir, exist_ok=True)

def _is_excluded(eco_code):
    if eco_code in ECO_EXCLUDE:
        return True
    name = eco_results.get(eco_code, {}).get('name', '')
    return any(ex.lower() in name.lower() or ex.lower() == eco_code.lower()
               for ex in ECO_EXCLUDE)

ALL_ECO_CODES = sorted(c for c in eco_results.keys() if not _is_excluded(c))
n_eco_total   = len(ALL_ECO_CODES)
n_cols        = 3
n_rows        = int(np.ceil(n_eco_total / n_cols))

print(f"  Ecoregions (exkl. {ECO_EXCLUDE}): {ALL_ECO_CODES}")

def get_eco_color(eco_code):
    return eco_results.get(eco_code, {}).get('hex_color', '#808080')

def get_eco_name(eco_code):
    return eco_results.get(eco_code, {}).get('name', eco_code)

def intensify_color(hex_color, fc_val):
    rgb = mcolors.to_rgb(hex_color)
    h, s, v = colorsys.rgb_to_hsv(*rgb)
    v_scale = {1: 1.0, 2: 0.82, 3: 0.65, 4: 0.50}
    return colorsys.hsv_to_rgb(h, s, v * v_scale.get(fc_val, 1.0))

def plot_recovery_panel(ax, traj, color):
    fire_year_cover = traj[5]
    if fire_year_cover <= 0 or np.isnan(fire_year_cover):
        return False
    traj_post = traj[post_fire_indices] / fire_year_cover
    ax.plot(POST_FIRE_YEARS, traj_post, marker='o', linewidth=2.5, color=color)
    ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5,
               alpha=0.7, label='Fire Year (= 1.0)')
    ax.set_xlabel('Years After First Fire', fontsize=9)
    ax.set_ylabel('Norm. Woody Cover', fontsize=9)
    ax.set_xlim(-0.3, 10.3)
    ax.set_ylim(0.5, 1.5)
    ax.set_xticks(POST_FIRE_YEARS)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)
    return True

print(f"  ✓ Helper-Funktionen definiert, {n_eco_total} Ecoregionen")


2h. RECOVERY-ANALYSE: Definitionen
----------------------------------------------------------------------
  Ecoregions (exkl. {'OUT'}): ['Alpine', 'Anatolian', 'Arctic', 'Atlantic', 'BlackSea', 'Boreal', 'Continental', 'Mediterranean', 'Pannonian', 'Steppic']
  ✓ Helper-Funktionen definiert, 10 Ecoregionen


HIER BEGINNT VIS

In [49]:
# =====================================================
# G1: Annual Burned Area (entire study area)
# =====================================================
if len(burned_area_total) > 0:
    print("\nPlot G1: Annual Burned Area...")

    fig, ax = plt.subplots(figsize=(14, 6))

    # Normalisiere Burned_km2 auf [0, 1] für die Farbskala
    norm = plt.Normalize(burned_area_total['Burned_km2'].min(), 
                        burned_area_total['Burned_km2'].max())
    colors = plt.cm.YlOrRd(norm(burned_area_total['Burned_km2']))

    bars = ax.bar(burned_area_total['Year'], burned_area_total['Burned_km2'],
            color=colors,
            alpha=0.8, edgecolor='darkred', linewidth=0.5)

    # Zahlen horizontal ausrichten (ganzzahlig)
    ax.bar_label(bars, label_type='edge', rotation=0, padding=3, fontsize=6, fmt='%.0f')

    mean_val = burned_area_total['Burned_km2'].mean()
    ax.axhline(y=mean_val, color='black', linestyle='--', linewidth=1.5,
               label=f'Mean: {mean_val:,.0f} km²/yr')
    ax.text

    ax.set_xlabel('Year', fontsize=12)
    ax.set_xlim(years_burned[0] - 0.5, years_burned[-1] + 0.5)
    ax.set_ylabel('Burned Area (km²)', fontsize=12)
    ax.set_title('Annual Burned Area – Entire Study Area', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    

    plt.tight_layout()
    plt.savefig(os.path.join(general_plots_dir, "burned_area_annual_total_alt.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ burned_area_annual_total_alt.png")

    # CSV
    csv_path = os.path.join(general_csv_dir, "burned_area_annual_total.csv")
    burned_area_total.to_csv(csv_path, index=False)
    print(f"  ✓ {csv_path}")


Plot G1: Annual Burned Area...
  ✓ burned_area_annual_total_alt.png
  ✓ D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\09_Results\General\csv\burned_area_annual_total.csv


In [54]:
# =====================================================
# G1: Annual Burned Area (entire study area)
# =====================================================
if len(burned_area_total) > 0:
    print("\nPlot G1: Annual Burned Area...")

    fig, ax = plt.subplots(figsize=(14, 6))

    # Normalisiere Burned_km2 auf [0, 1] für die Farbskala
    norm = plt.Normalize(burned_area_total['Burned_km2'].min(), 
                        burned_area_total['Burned_km2'].max())
    colors = plt.cm.YlOrRd(norm(burned_area_total['Burned_km2']))

    bars = ax.bar(burned_area_total['Year'], burned_area_total['Burned_km2'],
            color=colors,
            alpha=0.8, edgecolor='darkred', linewidth=0.5)

    # Zahlen horizontal ausrichten (ganzzahlig)
    ax.bar_label(bars, label_type='edge', rotation=0, padding=3, fontsize=6, fmt='%.0f')

    mean_val = burned_area_total['Burned_km2'].mean()
    cumulative_val = burned_area_total['Burned_km2'].sum()
    
    ax.axhline(y=mean_val, color='black', linestyle='--', linewidth=1.5)

    ax.set_xlabel('Year', fontsize=12)
    ax.set_xlim(years_burned[0] - 0.5, years_burned[-1] + 0.5)
    ax.set_ylabel('Burned Area (km²)', fontsize=12)
    ax.set_title('Annual Burned Area – Entire Study Area', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Statistics box top-right
    stats_text = (f"-------- Mean: {mean_val:,.0f} km²/yr\n"
                  f"Cumulative: {cumulative_val:,.0f} km²")
    ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
            fontsize=10, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()
    plt.savefig(os.path.join(general_plots_dir, "burned_area_annual_total.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ burned_area_annual_total.png")

    # CSV
    csv_path = os.path.join(general_csv_dir, "burned_area_annual_total.csv")
    burned_area_total.to_csv(csv_path, index=False)
    print(f"  ✓ {csv_path}")


Plot G1: Annual Burned Area...
  ✓ burned_area_annual_total.png
  ✓ D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\09_Results\General\csv\burned_area_annual_total.csv


In [ ]:

# =====================================================
# G2: Fire Season Length by Fire Count
# =====================================================
if season_by_fc:
    print("\nPlot G2: Fire Season Length by Fire Count...")

    fig, ax = plt.subplots(figsize=(14, 8))

    has_raw_data = all(len(season_by_fc[c].get('values', [])) > 0
                       for c in season_by_fc.keys())

    if has_raw_data:
        bp_data   = [season_by_fc[c]['values'] for c in sorted(season_by_fc.keys())]
        bp_labels = [f"{c} Fires\n(n={season_by_fc[c]['n']:,})"
                     for c in sorted(season_by_fc.keys())]
        bp = ax.boxplot(bp_data, labels=bp_labels, patch_artist=True)
        colors_gradient = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(bp_data)))
        for patch, color in zip(bp['boxes'], colors_gradient):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
    else:
        print("  ⚠️  No raw data — showing mean values as bar chart")
        fc_vals   = sorted(season_by_fc.keys())
        means     = [season_by_fc[c]['mean'] for c in fc_vals]
        bp_labels = [f"{c} Fires\n(n={season_by_fc[c]['n']:,})" for c in fc_vals]
        x_pos     = np.arange(len(fc_vals))
        colors_gradient = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(fc_vals)))
        ax.bar(x_pos + 1, means, color=colors_gradient, alpha=0.7,
               edgecolor='black', linewidth=0.8)
        ax.set_xticks(x_pos + 1)
        ax.set_xticklabels(bp_labels)

    for i, fc_val in enumerate(sorted(season_by_fc.keys())):
        y_pos_ann = season_by_fc[fc_val]['mean']
        ax.text(i + 1, y_pos_ann + 0.05, f'{y_pos_ann:.1f}',
                ha='center', va='bottom', fontweight='bold', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    ax.set_xlabel('Number of Fire Events', fontsize=12)
    ax.set_ylabel('Fire Season Length (Months)', fontsize=12)
    ax.set_title('Fire Season Length by Number of intraannual Fire Events',
                 fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(general_plots_dir, "fire__season_length_by_count.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ fire_season_length_by_count.png")

    # CSV
    rows_sfc = []
    for fc_val, fc_data in sorted(season_by_fc.items()):
        rows_sfc.append({
            'Fire_Count':          fc_val,
            'N_Pixels':            fc_data['n'],
            'Mean_Season_Length':   fc_data['mean'],
        })
    csv_path = os.path.join(general_csv_dir, "season_length_by_fire_count.csv")
    pd.DataFrame(rows_sfc).to_csv(csv_path, index=False)
    print(f"  ✓ {csv_path}")

print(f"\n✓ General visualizations complete → {general_plots_dir}")

In [24]:
# =====================================================
# G3: Fire Month Distribution (All Fire Events)
# =====================================================
print("\nPlot G3: Fire Month Distribution...")

month_csv = os.path.join(general_csv_dir, "fire_month_distribution.csv")

# --- Try loading from cache ---
if os.path.exists(month_csv):
    print("  Loading fire month distribution from cache...")
    month_df = pd.read_csv(month_csv)
    month_counts = month_df['Fire_Events'].values
else:
    print("  Computing fire month distribution from DOY bands...")
    
    # --- Load DOY bands (doy_1, doy_2, doy_3, doy_4) for all years ---
    doy_band_indices = []
    for year in years_burned:
        base_idx = 1 + (year - MBA_START_YEAR) * len(band_structure)
        doy_band_indices.extend([
            base_idx + 2,  # doy_1
            base_idx + 3,  # doy_2
            base_idx + 4,  # doy_3
            base_idx + 5,  # doy_4
        ])

    with rasterio.open(burned_path) as src:
        doy_all = src.read(doy_band_indices)

    print(f"    ✓ DOY bands loaded: {doy_all.shape}")

    # --- Convert DOY to Month ---
    def doy_to_month(doy):
        doy = np.asarray(doy, dtype=np.float32)
        if np.isscalar(doy):
            if doy == 0:
                return 0
            return int((doy - 1) // 30.44 + 1)
        else:
            result = np.zeros_like(doy, dtype=np.uint8)
            valid = doy > 0
            result[valid] = ((doy[valid] - 1) // 30.44 + 1).astype(np.uint8)
            return result

    # --- Count fire events per month ---
    month_counts = np.zeros(12, dtype=np.int64)
    for i in range(doy_all.shape[0]):
        doy_band = doy_all[i]
        months = doy_to_month(doy_band)
        for m in range(1, 13):
            month_counts[m - 1] += np.sum(months == m)

    print(f"    Fire events per month: {dict(zip(range(1, 13), month_counts))}")
    
    # Save to CSV for future use
    months_abbr = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    month_df = pd.DataFrame({
        'Month': months_abbr,
        'Month_Num': range(1, 13),
        'Fire_Events': month_counts,
        'Percent': (month_counts / month_counts.sum() * 100).round(2)
    })
    month_df.to_csv(month_csv, index=False)
    print(f"    ✓ Saved to cache: {month_csv}")

# --- Bar Chart (same code for both cache/fresh) ---
month_counts = month_df['Fire_Events'].values
months_abbr = month_df['Month'].values
total_events = month_counts.sum()

fig, ax = plt.subplots(figsize=(14, 7))
x_pos = np.arange(12)

# Normalize month_counts to [0, 1] for colormap
# Max fire count → 1 (red), Min fire count → 0 (green)
normalized = (month_counts - month_counts.min()) / (month_counts.max() - month_counts.min())
colors = plt.cm.YlOrRd(normalized)  # 0=green, 0.5=yellow, 1=red

bars = ax.bar(x_pos, month_counts, color=colors, edgecolor='black', linewidth=1.2, alpha=0.85)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Month', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Fire Events', fontsize=12, fontweight='bold')
ax.set_title('Temporal Distribution of Fire Events by Month\n(All Fires 2001–2025, including Intra- & Inter-Annual)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(months_abbr, fontsize=11)
ax.grid(True, axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, month_counts.max() * 1.1)

max_month = months_abbr[np.argmax(month_counts)]
min_month = months_abbr[np.argmin(month_counts)]
stats_text = (f"Total Fire Events: {total_events:,}\n"
              f"Mean per Month: {total_events / 12:,.0f}\n"
              f"Peak: {max_month} ({month_counts.max():,})\n"
              f"Lowest: {min_month} ({month_counts.min():,})")
ax.text(0.98, 0.97, stats_text, transform=ax.transAxes, 
        fontsize=10, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(general_plots_dir, "fire_month_distribution.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ fire_month_distribution.png")


Plot G3: Fire Month Distribution...
  Loading fire month distribution from cache...
  ✓ fire_month_distribution.png


In [26]:
# =====================================================
# G4: Fire Month Distribution by Ecoregion (Panel)
# =====================================================
print("\nPlot G4: Fire Month Distribution by Ecoregion (Panel)...")

month_eco_csv = os.path.join(general_csv_dir, "fire_month_distribution_by_ecoregion.csv")

# --- Try loading from cache ---
if os.path.exists(month_eco_csv):
    print("  Loading fire month distribution by ecoregion from cache...")
    month_eco_df = pd.read_csv(month_eco_csv)
else:
    print("  Computing fire month distribution by ecoregion from DOY bands (block-wise)...")
    
    # --- Ensure eco_raster is loaded ---
    if not isinstance(globals().get('eco_raster'), np.ndarray):
        with rasterio.open(ecoregion_raster_path) as src:
            eco_raster = src.read(1)

    # --- Convert DOY to Month ---
    def doy_to_month(doy):
        doy = np.asarray(doy, dtype=np.float32)
        result = np.zeros_like(doy, dtype=np.uint8)
        valid = doy > 0
        result[valid] = ((doy[valid] - 1) // 30.44 + 1).astype(np.uint8)
        return result

    # --- Prepare band indices ---
    doy_band_indices = []
    for year in years_burned:
        base_idx = 1 + (year - MBA_START_YEAR) * len(band_structure)
        doy_band_indices.extend([
            base_idx + 2,  # doy_1
            base_idx + 3,  # doy_2
            base_idx + 4,  # doy_3
            base_idx + 5,  # doy_4
        ])

    # --- Count fire events per month per ecoregion (block-wise) ---
    month_counts_by_eco = {eco_id: np.zeros(12, dtype=np.int64) 
                           for eco_id in unique_eco_ids}

    print("  Processing raster in blocks...")
    with rasterio.open(burned_path) as src:
        # Iterate over blocks instead of loading entire array
        for ji, window in src.block_windows(1):
            # Read DOY bands for this block only
            doy_block = src.read(doy_band_indices, window=window)  # Much smaller
            eco_block = eco_raster[window.row_off:window.row_off+window.height,
                                    window.col_off:window.col_off+window.width]
            
            # Process all 100 DOY bands within this block
            for band_idx in range(doy_block.shape[0]):
                doy_band = doy_block[band_idx]
                months = doy_to_month(doy_band)
                
                # Count per ecoregion
                for eco_id in unique_eco_ids:
                    eco_mask = (eco_block == eco_id)
                    masked_months = months[eco_mask]
                    for m in range(1, 13):
                        month_counts_by_eco[eco_id][m - 1] += np.sum(masked_months == m)
            
            print(f"    ✓ Processed block {ji}")

    # --- Build DataFrame ---
    month_eco_rows = []
    months_abbr = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

    for eco_id in unique_eco_ids:
        eco_code = eco_mapping[eco_id]['code']
        month_counts = month_counts_by_eco[eco_id]
        
        for m in range(12):
            month_eco_rows.append({
                'Ecoregion': eco_code,
                'Month': months_abbr[m],
                'Month_Num': m + 1,
                'Fire_Events': month_counts[m]
            })
        
        print(f"    ✓ {eco_code}: {month_counts.sum():,} fire events")

    month_eco_df = pd.DataFrame(month_eco_rows)
    month_eco_df['Percent'] = month_eco_df.groupby('Ecoregion')['Fire_Events'].transform(
        lambda x: (x / x.sum() * 100).round(2) if x.sum() > 0 else 0
    )
    month_eco_df.to_csv(month_eco_csv, index=False)
    print(f"    ✓ Saved to cache: {month_eco_csv}")

# --- Plot: Panel with one bar chart per ecoregion ---
n_eco_g4 = len(ALL_ECO_CODES)
n_cols_g4 = 3
n_rows_g4 = int(np.ceil(n_eco_g4 / n_cols_g4))

fig, axes = plt.subplots(n_rows_g4, n_cols_g4, figsize=(16, 4.5 * n_rows_g4))
axes_flat = axes.flatten()

months_abbr = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for idx, eco_code in enumerate(ALL_ECO_CODES):
    ax = axes_flat[idx]
    eco_data = month_eco_df[month_eco_df['Ecoregion'] == eco_code]
    
    if len(eco_data) == 0:
        ax.axis('off')
        continue
    
    month_counts = eco_data['Fire_Events'].values
    x_pos = np.arange(12)
    
    # Normalize colors: min=green, max=red for THIS ecoregion
    if month_counts.max() > 0:
        normalized = (month_counts - month_counts.min()) / (month_counts.max() - month_counts.min() + 1e-6)
    else:
        normalized = np.zeros(12)
    colors = plt.cm.RdYlGn_r(normalized)
    
    bars = ax.bar(x_pos, month_counts, color=colors, edgecolor='black', linewidth=1, alpha=0.85)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height):,}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Title with color border
    eco_name = get_eco_name(eco_code)
    eco_color = get_eco_color(eco_code)
    ax.set_title(f'{eco_code} – {eco_name}', fontsize=10, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_edgecolor(eco_color)
        spine.set_linewidth(2)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(months_abbr, fontsize=8, rotation=45)
    ax.set_ylabel('Fire Events', fontsize=9)
    ax.grid(True, axis='y', alpha=0.2, linestyle='--')
    if month_counts.max() > 0:
        ax.set_ylim(0, month_counts.max() * 1.15)

# Disable unused subplots
for idx in range(n_eco_g4, len(axes_flat)):
    axes_flat[idx].axis('off')

plt.suptitle('Temporal Distribution of Fire Events by Month and Ecoregion\n(All Fires 2001–2025, including Intra- & Inter-Annual)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.savefig(os.path.join(general_plots_dir, "fire_month_distribution_by_ecoregion_panel.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ fire_month_distribution_by_ecoregion_panel.png")


Plot G4: Fire Month Distribution by Ecoregion (Panel)...
  Computing fire month distribution by ecoregion from DOY bands (block-wise)...
  Processing raster in blocks...
    ✓ Processed block (0, 0)
    ✓ Processed block (0, 1)
    ✓ Processed block (0, 2)
    ✓ Processed block (0, 3)
    ✓ Processed block (0, 4)
    ✓ Processed block (0, 5)
    ✓ Processed block (0, 6)
    ✓ Processed block (0, 7)
    ✓ Processed block (0, 8)
    ✓ Processed block (0, 9)
    ✓ Processed block (0, 10)
    ✓ Processed block (0, 11)
    ✓ Processed block (0, 12)
    ✓ Processed block (0, 13)
    ✓ Processed block (0, 14)
    ✓ Processed block (0, 15)
    ✓ Processed block (0, 16)
    ✓ Processed block (0, 17)
    ✓ Processed block (0, 18)
    ✓ Processed block (0, 19)
    ✓ Processed block (0, 20)
    ✓ Processed block (1, 0)
    ✓ Processed block (1, 1)
    ✓ Processed block (1, 2)
    ✓ Processed block (1, 3)
    ✓ Processed block (1, 4)
    ✓ Processed block (1, 5)
    ✓ Processed block (1, 6)
    ✓ 

In [ ]:
# ============================================================
# 4.2a  ECOREGION — Trajectory Plots (E1–E3)
# ============================================================
print("=" * 70)
print("4.2a  ECOREGION TRAJECTORY PLOTS")
print("=" * 70)

# =====================================================
# E1: All Ecoregion Trajectories (1 Fire) — combined
# =====================================================
print("\nPlot E1: All Ecoregion Trajectories (1 Fire)...")

fig, ax = plt.subplots(figsize=(14, 8))

for eco_code in sorted(ALL_ECO_CODES):
    data = eco_results[eco_code]
    color = data["hex_color"]
    traj = np.array(data["trajectory"])

    ax.plot(
        rel_years,
        traj,
        marker="o",
        linewidth=2.5,
        label=f"{eco_code} (n={data['n_pixels']:,})",
        color=color,
        alpha=0.9,
    )

ax.axvline(x=0, color="red", linestyle="--", linewidth=1.8, alpha=0.6, label="Fire Year")
ax.axvline(x=1, color="red", linestyle="--", linewidth=1.8, alpha=0.6)

ax.set_xlabel("Years Relative to Fire Event", fontsize=12)
ax.set_ylabel("Woody Cover (%)", fontsize=12)
ax.set_xlim(-5.5, 10.5)
#ax.set_ylim(0, 100)
#ax.set_yticks(np.arange(0, 101, 10))
ax.grid(True, alpha=0.3)

# More title spacing (pad) and room on top.
ax.set_title(
    "Post-Fire Woody Cover Trajectories by Ecoregion\n(Single Fire Events Only)",
    fontsize=14,
    fontweight="bold",
    pad=20,
)

ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.subplots_adjust(top=0.88)

plt.savefig(
    os.path.join(eco_plots_dir, "eco_trajectories_1fire_all.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.close()
print("  ✓ eco_trajectories_1fire_all.png")

# =====================================================
# E2: Subplots per Ecoregion (1-4 Fires, separate panels)
# =====================================================
print("\nPlot E2: Subplots per Ecoregion (1-4 Fires)...")

panel_cfgs = [
    (1, "eco_trajectories_1fire_panel.png", "Single Fire Events"),
    (2, "eco_trajectories_2fires_panel.png", "2 Fire Events"),
    (3, "eco_trajectories_3fires_panel.png", "3 Fire Events"),
    (4, "eco_trajectories_4fires_panel.png", "4 Fire Events"),
]

for fc_val, out_name, panel_label in panel_cfgs:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes_flat = np.atleast_1d(axes).ravel()

    for i, eco_code in enumerate(ALL_ECO_CODES):
        ax = axes_flat[i]
        color = get_eco_color(eco_code)
        eco_name = get_eco_name(eco_code)

        if fc_val == 1:
            data = eco_results.get(eco_code)
        else:
            data = eco_firecount_results.get(eco_code, {}).get(fc_val)

        if data is None:
            ax.axis("off")
            ax.text(
                0.5,
                0.5,
                f"{eco_code}\nno data",
                ha="center",
                va="center",
                fontsize=9,
                color="lightgray",
                transform=ax.transAxes,
            )
            continue

        traj = np.array(data["trajectory"])
        n_pixels = data["n_pixels"]

        if len(traj) < len(rel_years):
            ax.axis("off")
            ax.text(
                0.5,
                0.5,
                f"{eco_code}\ninvalid trajectory",
                ha="center",
                va="center",
                fontsize=9,
                color="lightgray",
                transform=ax.transAxes,
            )
            continue

        # No SD shading: line-only style.
        ax.plot(rel_years, traj, marker="o", linewidth=2.3, color=color, alpha=0.95)
        ax.axvline(x=0, color="red", linestyle="--", linewidth=1.6, alpha=0.6)
        ax.axvline(x=1, color="red", linestyle="--", linewidth=1.6, alpha=0.6)

        ax.set_title(f"{eco_code} - {eco_name}\n(n={n_pixels:,})", fontsize=10, fontweight="bold")
        ax.set_xlabel("Years Relative to Fire", fontsize=9)
        ax.set_ylabel("Woody Cover (%)", fontsize=9)
        ax.set_xlim(-5.5, 10.5)
        ax.set_ylim(0, 100)
        ax.grid(True, alpha=0.3)

        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2)

    for j in range(n_eco_total, len(axes_flat)):
        axes_flat[j].axis("off")

    plt.suptitle(
        f"Woody Cover Trajectories by Ecoregion\n({panel_label})",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.subplots_adjust(top=0.90)

    plt.savefig(
        os.path.join(eco_plots_dir, out_name),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()
    print(f"  ✓ {out_name}")

# =====================================================
# E3: Ecoregion x Fire Count (combined panel style)   
# =====================================================
print("\nPlot E3: Ecoregion x Fire Count...")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes_flat = np.atleast_1d(axes).ravel()

for i, eco_code in enumerate(ALL_ECO_CODES):
    ax = axes_flat[i]
    base_color = get_eco_color(eco_code)
    eco_name = get_eco_name(eco_code)

    n_per_fc = {}
    has_any = False

    for fc_val in [1, 2, 3, 4]:
        data = eco_results.get(eco_code) if fc_val == 1 else eco_firecount_results.get(eco_code, {}).get(fc_val)
        if data is None:
            continue

        traj = np.array(data["trajectory"])
        if len(traj) < len(rel_years):
            continue

        fc_color = intensify_color(base_color, fc_val)
        ax.plot(
            rel_years,
            traj,
            marker="o",
            linewidth=2.0,
            color=fc_color,
            label=f"{fc_val} Fire(s)",
            alpha=0.95,
        )
        n_per_fc[fc_val] = data["n_pixels"]
        has_any = True

    if not has_any:
        ax.axis("off")
        ax.text(
            0.5,
            0.5,
            f"{eco_code}\nno data",
            ha="center",
            va="center",
            fontsize=9,
            color="lightgray",
            transform=ax.transAxes,
        )
        continue

    ax.axvline(x=0, color="red", linestyle="--", linewidth=1.6, alpha=0.6, label="Fire Year")
    ax.axvline(x=1, color="red", linestyle="--", linewidth=1.6, alpha=0.6)

    n_str = "  ".join([f"n{k}={v:,}" for k, v in sorted(n_per_fc.items())])
    ax.set_title(f"{eco_code}\n{n_str}", fontsize=9, fontweight="bold")
    ax.set_xlabel("Years Relative to Fire", fontsize=9)
    ax.set_ylabel("Woody Cover (%)", fontsize=9)
    ax.set_xlim(-5.5, 10.5)
    #ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="lower right")

    for spine in ax.spines.values():
        spine.set_edgecolor(base_color)
        spine.set_linewidth(2)

for j in range(n_eco_total, len(axes_flat)):
    axes_flat[j].axis("off")

plt.suptitle(
    "Woody Cover Trajectories by Ecoregion\nAll Inter-Annual Fire Counts",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.subplots_adjust(top=0.90)

plt.savefig(
    os.path.join(eco_plots_dir, "eco_trajectories_by_fire_count.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.close()
print("  ✓ eco_trajectories_by_fire_count.png")

# --- CSV Exports ---
print("\nSaving Trajectory CSVs...")

rows = []
for eco_code in ALL_ECO_CODES:
    data = eco_results[eco_code]
    for i, ry in enumerate(rel_years):
        rows.append(
            {
                "Ecoregion": eco_code,
                "Ecoregion_Name": data["name"],
                "N_Pixels": data["n_pixels"],
                "Rel_Year": ry,
                "Woody_Cover_Mean": data["trajectory"][i],
                "Woody_Cover_Std": data["std"][i],
            }
        )

pd.DataFrame(rows).to_csv(
    os.path.join(eco_csv_dir, "trajectories_by_ecoregion_1fire.csv"),
    index=False,
)
print("  ✓ trajectories_by_ecoregion_1fire.csv")

rows = []
for eco_code, fc_data in eco_firecount_results.items():
    if _is_excluded(eco_code):
        continue
    for fc, data in fc_data.items():
        for i, ry in enumerate(rel_years):
            rows.append(
                {
                    "Ecoregion": eco_code,
                    "Fire_Count": fc,
                    "N_Pixels": data["n_pixels"],
                    "Rel_Year": ry,
                    "Woody_Cover_Mean": data["trajectory"][i],
                    "Woody_Cover_Std": data["std"][i],
                }
            )

pd.DataFrame(rows).to_csv(
    os.path.join(eco_csv_dir, "trajectories_by_ecoregion_x_firecount.csv"),
    index=False,
)
print("  ✓ trajectories_by_ecoregion_x_firecount.csv")

print("\n✓ 4.2a complete")

4.2a  ECOREGION TRAJECTORY PLOTS

Plot E1: All Ecoregion Trajectories (1 Fire)...
  ✓ eco_trajectories_1fire_all.png

Plot E2: Subplots per Ecoregion (1-4 Fires)...
  ✓ eco_trajectories_1fire_panel.png
  ✓ eco_trajectories_2fires_panel.png
  ✓ eco_trajectories_3fires_panel.png
  ✓ eco_trajectories_4fires_panel.png

Plot E3: Ecoregion x Fire Count...
  ✓ eco_trajectories_by_fire_count.png

Saving Trajectory CSVs...
  ✓ trajectories_by_ecoregion_1fire.csv
  ✓ trajectories_by_ecoregion_x_firecount.csv

✓ 4.2a complete


In [30]:
# ============================================================
# 4.2b  ECOREGION — Fire Statistics (E5–E7)
# ============================================================
print("=" * 70)
print("4.2b  ECOREGION FIRE STATISTICS")
print("=" * 70)

# Load MBA bands on demand (for E6/E7 plots)
if mba_count is None:
    print("Loading MBA season/count bands...")
    with rasterio.open(burned_path) as src:
        count_band_indices  = [1 + (yr - MBA_START_YEAR) * len(band_structure) + 1
                               for yr in years_burned]
        season_band_indices = [1 + (yr - MBA_START_YEAR) * len(band_structure) + 11
                               for yr in years_burned]
        mba_count  = src.read(count_band_indices)
        mba_season = src.read(season_band_indices)
    print(f"  mba_count={mba_count.shape}, mba_season={mba_season.shape}")

    if not season_by_fc:
        count_flat  = mba_count.flatten()
        season_flat = mba_season.flatten()
        valid_mask  = (count_flat > 0) & (season_flat > 0)
        count_valid  = count_flat[valid_mask].astype(np.int32)
        season_valid = season_flat[valid_mask].astype(np.float32)
        season_by_fc = {}
        for fc_val in np.unique(count_valid):
            fc_mask = (count_valid == fc_val)
            vals = season_valid[fc_mask]
            season_by_fc[int(fc_val)] = {
                'values': vals, 'n': int(len(vals)), 'mean': float(vals.mean()),
            }
        print(f"  season_by_fc reconstructed: {sorted(season_by_fc.keys())}")

4.2b  ECOREGION FIRE STATISTICS
Loading MBA season/count bands...
  mba_count=(25, 9660, 10483), mba_season=(25, 9660, 10483)


In [ ]:
# =====================================================
# E5: Burned Area per Ecoregion (gestapelte Barplots)
# =====================================================
if len(burned_area_eco) > 0:
    print("\nPlot E5: Burned Area per Ecoregion (stacked bars)...")
    
    # Filter ab 2001 und entferne "Outside"
    burned_area_eco_2001 = burned_area_eco[
        (burned_area_eco['Year'] >= 2001) & 
        (burned_area_eco['Ecoregion'] != 'Outside')
    ].copy()
    pivot_abs = burned_area_eco_2001.pivot_table(
        index='Year', columns='Ecoregion', values='Burned_km2', fill_value=0)
    pivot_pct = burned_area_eco_2001.pivot_table(
        index='Year', columns='Ecoregion', values='Burned_Fraction', fill_value=0) * 100
    
    # Spalten alphabetisch sortieren, dann umkehren (für Stacking: Z-A von unten nach oben)
    pivot_abs = pivot_abs[sorted(pivot_abs.columns)[::-1]]
    pivot_pct = pivot_pct[sorted(pivot_pct.columns)[::-1]]
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Farbschema
    colors = {eco: get_eco_color(eco) for eco in sorted(pivot_abs.columns, reverse=True)}
    colors_list = [colors.get(col, '#808080') for col in pivot_abs.columns]
    
    # Plot 1: Absolute Werte (gestapelt)
    pivot_abs.plot(kind='bar', stacked=True, ax=axes[0], color=colors_list, 
                   width=0.7, edgecolor='black', linewidth=0.5, legend=False)
    
    axes[0].set_xlabel('Year', fontsize=12)
    axes[0].set_ylabel('Burned Area (km²)', fontsize=12)
    axes[0].set_title('Annual Burned Area per Ecoregion (absolute, stacked)',
                      fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # X-Achsen-Label: nur jedes 2. Jahr anzeigen
    x_labels = [str(int(year)) if i % 2 == 0 else '' for i, year in enumerate(pivot_abs.index)]
    axes[0].set_xticklabels(x_labels, rotation=45, ha='right')
    
    # Plot 2: Prozentuale Werte (gestapelt)
    pivot_pct.plot(kind='bar', stacked=True, ax=axes[1], color=colors_list, 
                   width=0.7, edgecolor='black', linewidth=0.5, legend=False)
    
    axes[1].set_xlabel('Year', fontsize=12)
    axes[1].set_ylabel('Burned Area (% of Ecoregion)', fontsize=12)
    axes[1].set_title('Annual Burned Area per Ecoregion (% of Ecoregion area, stacked)',
                      fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # X-Achsen-Label: nur jedes 2. Jahr anzeigen
    axes[1].set_xticklabels(x_labels, rotation=45, ha='right')
    
    # Legend in alphabetischer Reihenfolge (A-Z)
    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(handles[::-1], labels[::-1], title='Ecoregion',
                   bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
    # axes[1].legend(handles[::-1], labels[::-1], title='Ecoregion',
    #                bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
    
    #plt.suptitle('Annual Burned Area by Ecoregion (2001–2025)', 
    #             fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(os.path.join(eco_plots_dir, "burned_area_annual_by_ecoregion.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ burned_area_annual_by_ecoregion.png")
else:
    print("\n  ⚠ burned_area_eco empty — skipping E5")

In [ ]:

# =====================================================
# E6: Fire Season Length (2 Subplots)
# =====================================================
if len(fire_stats_summary) > 0:
    print("\nPlot E6: Fire Season Length (2 Subplots)...")

    ECO_PLOT_ORDER = sorted([
        eco for eco in fire_stats_summary['Ecoregion'].values if eco != 'Outside'])
    ECO_REV = ECO_PLOT_ORDER[::-1]
    y_pos_e6 = np.arange(len(ECO_PLOT_ORDER))
    n_ecos_e6 = len(ECO_PLOT_ORDER)
    eco_code_to_id = {v['code']: k for k, v in eco_mapping.items()}

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # E6a) Heatmap Frequency
    ax = axes[0]
    hm_season = np.full((n_ecos_e6, 12), np.nan)
    for i, eco_code in enumerate(ECO_PLOT_ORDER):
        eco_id = eco_code_to_id.get(eco_code)
        if eco_id is None:
            continue
        vals = mba_season[:, eco_raster == eco_id].flatten()
        vals = vals[(vals > 0) & (~np.isnan(vals))]
        if len(vals) == 0:
            continue
        uv, cn = np.unique(vals.astype(int), return_counts=True)
        total = len(vals)
        for u, c in zip(uv, cn):
            if 1 <= u <= 12:
                hm_season[i, u - 1] = c / total * 100

    cmap_s = plt.cm.YlOrRd.copy()
    cmap_s.set_bad(color='white')
    im = ax.imshow(np.ma.masked_invalid(hm_season), cmap=cmap_s, aspect='auto',
                   vmin=0, vmax=100, origin='upper')
    for ri in range(n_ecos_e6):
        for cj in range(12):
            val = hm_season[ri, cj]
            if not np.isnan(val) and val > 0:
                tc = 'white' if val > 60 else 'black'
                ax.text(cj, ri, f"{val:.2f}%", ha='center', va='center', fontsize=6, color=tc)
    ax.set_xticks(range(12))
    ax.set_xticklabels(range(1, 13))
    ax.set_yticks(range(n_ecos_e6))
    ax.set_yticklabels(ECO_PLOT_ORDER)
    ax.set_xlabel('Fire Season Length (Months)', fontsize=11)
    ax.set_title('Season Length Frequency\n(% of pixel-years per ecoregion)',
                 fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='%', shrink=0.8)

        # E6b) Temporal Trend Heatmap — Einheitlicher YlOrRd Gradient
    ax = axes[1]
    years_2001 = np.array(years_burned)[np.array(years_burned) >= 2001]
    idx_2001 = np.where(np.array(years_burned) >= 2001)[0]
    
    # Erstelle Matrix für max season length
    hm_temporal_season = np.full((n_ecos_e6, len(years_2001)), np.nan)
    for i, eco_code in enumerate(ECO_PLOT_ORDER):
        eco_id = eco_code_to_id.get(eco_code)
        if eco_id is None:
            continue
        for j, idx in enumerate(idx_2001):
            v = mba_season[idx, eco_raster == eco_id].flatten()
            v = v[(v > 0) & (~np.isnan(v))]
            if len(v) > 0:
                hm_temporal_season[i, j] = float(np.max(v))
    
    # Globale Normalisierung [0, 12] → [0, 1]
    hm_temporal_season_norm = np.full_like(hm_temporal_season, np.nan)
    for i in range(n_ecos_e6):
        for j in range(len(years_2001)):
            if not np.isnan(hm_temporal_season[i, j]):
                val = hm_temporal_season[i, j]
                # Normalisiere global: 1 → 0.083, 12 → 1.0
                hm_temporal_season_norm[i, j] = max(0, min(1, (val - 1) / 11))
    
    # Verwende einheitlichen YlOrRd colormap
    cmap_seasonal = plt.cm.YlOrRd
    im_s = ax.imshow(np.ma.masked_invalid(hm_temporal_season_norm), cmap=cmap_seasonal, 
                     aspect='auto', vmin=0, vmax=1, origin='upper')
    
    # Beschriftungen (ganzzahlig)
    for i in range(n_ecos_e6):
        for j in range(len(years_2001)):
            val = hm_temporal_season[i, j]
            if not np.isnan(val):
                tc = 'white' if hm_temporal_season_norm[i, j] > 0.6 else 'black'
                ax.text(j, i, f'{int(val)}', ha='center', va='center', 
                       fontsize=7, color=tc, fontweight='bold')
    
    ax.set_xticks(np.arange(0, len(years_2001), 2))
    ax.set_xticklabels(years_2001[::2], rotation=45)
    ax.set_yticks(range(n_ecos_e6))
    ax.set_yticklabels(ECO_PLOT_ORDER)
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Ecoregion', fontsize=11)
    ax.set_title('Temporal Trend: Fire Season Length Maximum\nper Ecoregion and Year',
                 fontsize=12, fontweight='bold')
    
    # Colorbar mit benutzerdefinierten Ticks für 1-12 Monate
    cbar = plt.colorbar(im_s, ax=ax, label='Season Length (Months)', shrink=0.8)
    # Setze Ticks für 1-12 Monate
    ticks_s = np.linspace(0, 1, 12)
    cbar.set_ticks(ticks_s)
    cbar.set_ticklabels([str(i) for i in range(1, 13)])
    
    #plt.suptitle('Fire Season Length Analysis', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(eco_plots_dir, "fire_season_length_analysis.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ fire_season_length_analysis.png")

    # # E6b) Temporal Trend Heatmap (Maximum pro Jahr pro Ecoregion) eco-farbgradient
    # ax = axes[1]
    # years_2001 = np.array(years_burned)[np.array(years_burned) >= 2001]
    # idx_2001 = np.where(np.array(years_burned) >= 2001)[0]
    
    # # Erstelle Matrix für max season length
    # hm_temporal_season = np.full((n_ecos_e6, len(years_2001)), np.nan)
    # for i, eco_code in enumerate(ECO_PLOT_ORDER):
    #     eco_id = eco_code_to_id.get(eco_code)
    #     if eco_id is None:
    #         continue
    #     for j, idx in enumerate(idx_2001):
    #         v = mba_season[idx, eco_raster == eco_id].flatten()
    #         v = v[(v > 0) & (~np.isnan(v))]
    #         if len(v) > 0:
    #             hm_temporal_season[i, j] = float(np.max(v))
    
    # # Normalisiere pro Ecoregion auf [0, 1]
    # hm_temporal_season_norm = np.full_like(hm_temporal_season, np.nan)
    # for i in range(n_ecos_e6):
    #     row = hm_temporal_season[i, :]
    #     valid = row[~np.isnan(row)]
    #     if len(valid) > 0:
    #         row_min, row_max = valid.min(), valid.max()
    #         if row_max > row_min:
    #             hm_temporal_season_norm[i, :] = (row - row_min) / (row_max - row_min)
    #         else:
    #             hm_temporal_season_norm[i, :] = 0.5
    
    # # Erstelle Farb-Matrix (RGB) — Interpolation von Weiß zu Ecoregion-Farbe
    # from matplotlib.colors import to_rgb
    # color_matrix = np.ones((*hm_temporal_season_norm.shape, 3))
    
    # for i, eco_code in enumerate(ECO_PLOT_ORDER):
    #     eco_color_hex = get_eco_color(eco_code)
    #     eco_color_rgb = to_rgb(eco_color_hex)
        
    #     for j in range(len(years_2001)):
    #         if not np.isnan(hm_temporal_season_norm[i, j]):
    #             val = hm_temporal_season_norm[i, j]
    #             # Interpolation: Weiß (1,1,1) → Eco-Farbe
    #             color_matrix[i, j, :] = [
    #                 1 - val * (1 - eco_color_rgb[0]),
    #                 1 - val * (1 - eco_color_rgb[1]),
    #                 1 - val * (1 - eco_color_rgb[2])
    #             ]
    
    # ax.imshow(color_matrix, aspect='auto', origin='upper')
    
    # # Beschriftungen (ganzzahlig)
    # for i in range(n_ecos_e6):
    #     for j in range(len(years_2001)):
    #         val = hm_temporal_season[i, j]
    #         if not np.isnan(val):
    #             tc = 'white' if hm_temporal_season_norm[i, j] > 0.6 else 'black'
    #             ax.text(j, i, f'{int(val)}', ha='center', va='center', 
    #                    fontsize=7, color=tc, fontweight='bold')
    
    # ax.set_xticks(np.arange(0, len(years_2001), 2))
    # ax.set_xticklabels(years_2001[::2], rotation=45)
    # ax.set_yticks(range(n_ecos_e6))
    # ax.set_yticklabels(ECO_PLOT_ORDER)
    # ax.set_xlabel('Year', fontsize=11)
    # ax.set_ylabel('Ecoregion', fontsize=11)
    # ax.set_title('Temporal Trend: Fire Season Length Maximum\nper Ecoregion and Year',
    #             fontsize=12, fontweight='bold')

    # plt.suptitle('Fire Season Length Analysis', fontsize=15, fontweight='bold')
    # plt.tight_layout()
    # plt.savefig(os.path.join(eco_plots_dir, "fire_season_length_analysis.png"),
    #             dpi=300, bbox_inches='tight')
    # plt.close()
    # print(f"  ✓ fire_season_length_analysis.png")


In [ ]:

# =====================================================
# E7: Fire Count (2 Subplots)
# =====================================================
if len(fire_stats_summary) > 0:
    print("\nPlot E7: Fire Count (2 Subplots)...")

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # E7a) Heatmap Frequency
    ax = axes[0]
    hm_count = np.full((n_ecos_e6, 4), np.nan)
    for i, eco_code in enumerate(ECO_PLOT_ORDER):
        eco_id = eco_code_to_id.get(eco_code)
        if eco_id is None:
            continue
        vals = mba_count[:, eco_raster == eco_id].flatten()
        vals = vals[(vals > 0) & (~np.isnan(vals))]
        if len(vals) == 0:
            continue
        uv, cn = np.unique(vals.astype(int), return_counts=True)
        total = len(vals)
        for u, c in zip(uv, cn):
            if 1 <= u <= 4:
                hm_count[i, u - 1] = c / total * 100

    cmap_c = plt.cm.YlOrRd.copy()
    cmap_c.set_bad(color='white')
    im2 = ax.imshow(np.ma.masked_invalid(hm_count), cmap=cmap_c, aspect='auto',
                    vmin=0, vmax=100, origin='upper')
    for ri in range(n_ecos_e6):
        for cj in range(4):
            val = hm_count[ri, cj]
            if not np.isnan(val) and val > 0:
                tc = 'white' if val > 60 else 'black'
                ax.text(cj, ri, f"{val:.2f}%", ha='center', va='center', fontsize=8, color=tc)
    ax.set_xticks(range(4))
    ax.set_xticklabels(range(1, 5))
    ax.set_yticks(range(n_ecos_e6))
    ax.set_yticklabels(ECO_PLOT_ORDER)
    ax.set_xlabel('Number of Fire Events', fontsize=11)
    ax.set_title('Fire Count Frequency\n(% of pixel-years per ecoregion)',
                 fontsize=12, fontweight='bold')
    plt.colorbar(im2, ax=ax, label='%', shrink=0.8)

        # E7b) Temporal Trend Heatmap — Einheitlicher YlOrRd Gradient
    ax = axes[1]
    
    # Erstelle Matrix für max fire count
    hm_temporal_count = np.full((n_ecos_e6, len(years_2001)), np.nan)
    for i, eco_code in enumerate(ECO_PLOT_ORDER):
        eco_id = eco_code_to_id.get(eco_code)
        if eco_id is None:
            continue
        for j, idx in enumerate(idx_2001):
            v = mba_count[idx, eco_raster == eco_id].flatten()
            v = v[(v > 0) & (~np.isnan(v))]
            if len(v) > 0:
                hm_temporal_count[i, j] = float(np.max(v))
    
    # Globale Normalisierung [1, 4] → [0, 1]
    hm_temporal_count_norm = np.full_like(hm_temporal_count, np.nan)
    for i in range(n_ecos_e6):
        for j in range(len(years_2001)):
            if not np.isnan(hm_temporal_count[i, j]):
                val = hm_temporal_count[i, j]
                # Normalisiere global: 1 → 0.0, 4 → 1.0
                hm_temporal_count_norm[i, j] = max(0, min(1, (val - 1) / 3))
    
    # Verwende einheitlichen YlOrRd colormap
    cmap_counts = plt.cm.YlOrRd
    im_c = ax.imshow(np.ma.masked_invalid(hm_temporal_count_norm), cmap=cmap_counts, 
                     aspect='auto', vmin=0, vmax=1, origin='upper')
    
    # Beschriftungen (ganzzahlig)
    for i in range(n_ecos_e6):
        for j in range(len(years_2001)):
            val = hm_temporal_count[i, j]
            if not np.isnan(val):
                tc = 'white' if hm_temporal_count_norm[i, j] > 0.6 else 'black'
                ax.text(j, i, f'{int(val)}', ha='center', va='center', 
                       fontsize=7, color=tc, fontweight='bold')
    
    ax.set_xticks(np.arange(0, len(years_2001), 2))
    ax.set_xticklabels(years_2001[::2], rotation=45)
    ax.set_yticks(range(n_ecos_e6))
    ax.set_yticklabels(ECO_PLOT_ORDER)
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Ecoregion', fontsize=11)
    ax.set_title('Temporal Trend: Fire Count Maximum\nper Ecoregion and Year',
                 fontsize=12, fontweight='bold')
    
    # Colorbar mit benutzerdefinierten Ticks für 1-4 Fire Events
    cbar = plt.colorbar(im_c, ax=ax, label='Fire Count', shrink=0.8)
    # Setze Ticks für 1-4 Fire Events: 0.0, 0.333, 0.667, 1.0
    ticks_c = [0, 1/3, 2/3, 1.0]
    cbar.set_ticks(ticks_c)
    cbar.set_ticklabels(['1', '2', '3', '4'])
    
    #plt.suptitle('Fire Count Analysis', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(eco_plots_dir, "fire_count_analysis.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ fire_count_analysis.png")

#     # E7b) Temporal Trend Heatmap (Maximum pro Jahr pro Ecoregion) eco-farbgradient
#     ax = axes[1]
    
#     # Erstelle Matrix für max fire count
#     hm_temporal_count = np.full((n_ecos_e6, len(years_2001)), np.nan)
#     for i, eco_code in enumerate(ECO_PLOT_ORDER):
#         eco_id = eco_code_to_id.get(eco_code)
#         if eco_id is None:
#             continue
#         for j, idx in enumerate(idx_2001):
#             v = mba_count[idx, eco_raster == eco_id].flatten()
#             v = v[(v > 0) & (~np.isnan(v))]
#             if len(v) > 0:
#                 hm_temporal_count[i, j] = float(np.max(v))
    
#     # GLOBALE Normalisierung für Fire Count 1-4 (nicht pro Reihe!)
#     hm_temporal_count_norm = np.full_like(hm_temporal_count, np.nan)
#     for i in range(n_ecos_e6):
#         for j in range(len(years_2001)):
#             if not np.isnan(hm_temporal_count[i, j]):
#                 val = hm_temporal_count[i, j]
#                 # Normalisiere global: 1 → 0, 2 → 0.333, 3 → 0.667, 4 → 1.0
#                 hm_temporal_count_norm[i, j] = max(0, min(1, (val - 1) / 3))
    
#     # Erstelle Farb-Matrix (RGB) — Interpolation von Weiß zu Ecoregion-Farbe
#     color_matrix_c = np.ones((*hm_temporal_count_norm.shape, 3))
    
#     for i, eco_code in enumerate(ECO_PLOT_ORDER):
#         eco_color_hex = get_eco_color(eco_code)
#         eco_color_rgb = to_rgb(eco_color_hex)
        
#         for j in range(len(years_2001)):
#             if not np.isnan(hm_temporal_count_norm[i, j]):
#                 val = hm_temporal_count_norm[i, j]
#                 # Interpolation: Weiß (1,1,1) → Eco-Farbe
#                 color_matrix_c[i, j, :] = [
#                     1 - val * (1 - eco_color_rgb[0]),
#                     1 - val * (1 - eco_color_rgb[1]),
#                     1 - val * (1 - eco_color_rgb[2])
#                 ]
    
#     ax.imshow(color_matrix_c, aspect='auto', origin='upper')
    
#     # Beschriftungen (ganzzahlig)
#     for i in range(n_ecos_e6):
#         for j in range(len(years_2001)):
#             val = hm_temporal_count[i, j]
#             if not np.isnan(val):
#                 tc = 'white' if hm_temporal_count_norm[i, j] > 0.6 else 'black'
#                 ax.text(j, i, f'{int(val)}', ha='center', va='center', 
#                        fontsize=7, color=tc, fontweight='bold')
    
#     ax.set_xticks(np.arange(0, len(years_2001), 2))
#     ax.set_xticklabels(years_2001[::2], rotation=45)
#     ax.set_yticks(range(n_ecos_e6))
#     ax.set_yticklabels(ECO_PLOT_ORDER)
#     ax.set_xlabel('Year', fontsize=11)
#     ax.set_ylabel('Ecoregion', fontsize=11)
#     ax.set_title('Temporal Trend: Fire Count Maximum\nper Ecoregion and Year',
#                 fontsize=12, fontweight='bold')
    
#     plt.suptitle('Fire Count Analysis', fontsize=15, fontweight='bold')
#     plt.tight_layout()
#     plt.savefig(os.path.join(eco_plots_dir, "fire_count_analysis.png"),
#                 dpi=300, bbox_inches='tight')
#     plt.close()
#     print(f"  ✓ fire_count_analysis.png")
# else:
#     print("\n  ⚠ fire_stats_summary empty — skipping E6/E7")


# --- CSV Exports ---
print("\nSaving Fire Stats CSVs...")

if len(burned_area_eco) > 0:
    burned_area_eco.to_csv(os.path.join(eco_csv_dir, "burned_area_by_ecoregion.csv"), index=False)
    print(f"  ✓ burned_area_by_ecoregion.csv")

if len(fire_count_eco) > 0:
    fire_count_eco.to_csv(os.path.join(eco_csv_dir, "fire_count_distribution_by_ecoregion.csv"), index=False)
    print(f"  ✓ fire_count_distribution_by_ecoregion.csv")

if len(fire_stats_summary) > 0:
    fire_stats_summary.to_csv(os.path.join(eco_csv_dir, "fire_statistics_summary_per_ecoregion.csv"), index=False)
    season_stats.to_csv(os.path.join(eco_csv_dir, "fire_season_length_by_ecoregion.csv"), index=False)
    print(f"  ✓ fire_statistics_summary_per_ecoregion.csv")
    print(f"  ✓ fire_season_length_by_ecoregion.csv")

print("\n✓ 4.2b complete")

In [28]:
# ============================================================
# 4.2c  ECOREGION — Recovery Plots (E8–E10)
# ============================================================
print("=" * 70)
print("4.2c  ECOREGION RECOVERY PLOTS")
print("=" * 70)


# =====================================================
# E8: Recovery Normalized Panels (per fire count 1–4 + combined)
# =====================================================
print("\nPlot E8: Recovery Normalized Panels...")

all_recovery_rows = []

for fc_val in [1, 2, 3, 4]:
    if fc_val == 1:
        source   = {c: d for c, d in eco_results.items() if not _is_excluded(c)}
        fc_label = "1_interannual_fire"
        fc_title = "1 Fire Year  (inter-annual Fire Count = 1)"
    else:
        source = {ec: fd[fc_val]
                  for ec, fd in eco_firecount_results.items()
                  if fc_val in fd and not _is_excluded(ec)}
        fc_label = f"{fc_val}_interannual_fires"
        fc_title = f"{fc_val} Fire Years  (inter-annual Fire Count = {fc_val})"

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes_flat = axes.flatten()

    for i, eco_code in enumerate(ALL_ECO_CODES):
        ax    = axes_flat[i]
        color = get_eco_color(eco_code)

        if eco_code not in source:
            ax.axis('off')
            ax.text(0.5, 0.5, f"{eco_code}\nno data",
                    ha='center', va='center', fontsize=9, color='lightgray',
                    transform=ax.transAxes)
            continue

        data     = source[eco_code]
        traj     = np.array(data['trajectory'])
        n_pixels = data['n_pixels']
        if len(traj) < 16:
            ax.axis('off'); continue

        fire_year_cover = traj[5]
        if fire_year_cover <= 0 or np.isnan(fire_year_cover):
            ax.axis('off'); continue

        ok = plot_recovery_panel(ax, traj, color)
        if ok:
            ax.set_title(f"{eco_code}  (n={n_pixels:,})", fontsize=10, fontweight='bold')
            for yr, idx in zip(POST_FIRE_YEARS, post_fire_indices):
                all_recovery_rows.append({
                    'Fire_Count_Interannual': fc_val,
                    'Ecoregion':             eco_code,
                    'Year_Relative':         yr,
                    'Cover_Normalized':      round(float(traj[idx]) / fire_year_cover, 4),
                    'Cover_Absolute':        round(float(traj[idx]), 2),
                    'N_Pixels':              n_pixels,
                })

    for j in range(n_eco_total, len(axes_flat)):
        axes_flat[j].axis('off')

    plt.suptitle(f'Post-Fire Woody Cover Recovery by Ecoregion\n{fc_title}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.subplots_adjust(top=0.88)
    plt.savefig(os.path.join(recovery_plots_dir,
                f"eco_recovery_normalized_{fc_label}_panel.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ eco_recovery_normalized_{fc_label}_panel.png")

# Combined plot: all fire counts per ecoregion
print("\n  Creating combined recovery plot...")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes_flat = axes.flatten()

for i, eco_code in enumerate(ALL_ECO_CODES):
    ax         = axes_flat[i]
    base_color = get_eco_color(eco_code)
    n_per_fc   = {}
    has_any    = False

    for fc_val in [1, 2, 3, 4]:
        data = eco_results.get(eco_code) if fc_val == 1 \
               else eco_firecount_results.get(eco_code, {}).get(fc_val)
        if data is None: continue
        traj = np.array(data['trajectory'])
        if len(traj) < 16: continue
        fire_year_cover = traj[5]
        if fire_year_cover <= 0 or np.isnan(fire_year_cover): continue

        traj_post = traj[post_fire_indices] / fire_year_cover
        fc_color  = intensify_color(base_color, fc_val)
        ax.plot(POST_FIRE_YEARS, traj_post, marker='o', linewidth=2,
                color=fc_color, label=f'{fc_val} Fire(s)', alpha=0.9)
        n_per_fc[fc_val] = data['n_pixels']
        has_any = True

    if not has_any:
        ax.axis('off')
        ax.text(0.5, 0.5, f"{eco_code}\nno data",
                ha='center', va='center', fontsize=9, color='lightgray',
                transform=ax.transAxes)
        continue

    ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, alpha=0.6,
               label='Fire Year (= 1.0)')
    n_str = '  '.join([f"n{k}={v:,}" for k, v in sorted(n_per_fc.items())])
    ax.set_title(f"{eco_code}\n{n_str}", fontsize=9, fontweight='bold')
    ax.set_xlabel('Years After First Fire', fontsize=9)
    ax.set_ylabel('Norm. Woody Cover', fontsize=9)
    ax.set_xlim(-0.3, 10.3); ax.set_ylim(0.5, 1.5)
    ax.set_xticks(POST_FIRE_YEARS)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)
    for spine in ax.spines.values():
        spine.set_edgecolor(base_color); spine.set_linewidth(2)

for j in range(n_eco_total, len(axes_flat)):
    axes_flat[j].axis('off')

plt.suptitle('Post-Fire Woody Cover Recovery by Ecoregion\n'
             'All Inter-Annual Fire Counts  (Year 0 = Fire Year = 1.0)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.subplots_adjust(top=0.90)
plt.savefig(os.path.join(recovery_plots_dir, "eco_recovery_normalized_combined_panel.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_recovery_normalized_combined_panel.png")

# =====================================================
# E9: Recovery Heatmap (pre-fire normalized)
# =====================================================
print("\nPlot E9: Recovery Heatmap (pre-fire normalized)...")

YEARS_AFTER   = list(range(0, 11))
post_fire_idx = list(range(5, 16))
pre_fire_idx  = list(range(0, 5))

eco_codes_hm = [c for c in ALL_ECO_CODES if c in eco_results]
n_years_hm   = len(YEARS_AFTER)
n_eco_hm     = len(eco_codes_hm)
hm_matrix    = np.full((n_years_hm, n_eco_hm), np.nan)
hm_n_pixels  = {}

for j, eco_code in enumerate(eco_codes_hm):
    data = eco_results[eco_code]
    traj = np.array(data['trajectory'])
    if len(traj) < 16: continue
    pre_fire_mean = np.nanmean(traj[pre_fire_idx])
    if pre_fire_mean <= 0 or np.isnan(pre_fire_mean): continue
    for i, idx in enumerate(post_fire_idx):
        hm_matrix[i, j] = traj[idx] / pre_fire_mean
    hm_n_pixels[eco_code] = data['n_pixels']

fig, ax = plt.subplots(figsize=(max(10, n_years_hm * 1.1), max(6, n_eco_hm * 0.8)))
cmap_hm = plt.cm.RdYlGn
vmin, vmax = 0.5, 1.3
norm_hm = plt.Normalize(vmin=vmin, vmax=vmax)
im = ax.imshow(hm_matrix.T, aspect='auto', cmap=cmap_hm, norm=norm_hm,
               origin='upper', interpolation='nearest')
ax.set_xticks(np.arange(n_years_hm))
ax.set_xticklabels([f'+{yr}' for yr in YEARS_AFTER], fontsize=9)
ax.set_xlabel('Years After Fire', fontsize=11)
ax.set_yticks(np.arange(n_eco_hm))
ax.set_yticklabels([f"{c}  (n={hm_n_pixels.get(c, 0):,})" for c in eco_codes_hm], fontsize=9)
ax.set_ylabel('Ecoregion', fontsize=11)

for i in range(n_eco_hm):
    for j in range(n_years_hm):
        val = hm_matrix[j, i]
        if not np.isnan(val):
            tc = 'white' if (val < 0.7 or val > 1.2) else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color=tc, fontweight='bold')

ax.axvline(x=0.5, color='black', linewidth=2, alpha=0.6, linestyle='--')
from mpl_toolkits.axes_grid1 import make_axes_locatable
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.1)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Norm. Woody Cover (Pre-Fire = 1.0)', fontsize=10)
cbar.ax.set_ylim(vmin, vmax)
cbar.ax.axhline(y=1.0, color='black', linewidth=1.5, linestyle='--')
cbar.ax.text(1.05, 1.0, '1.0', va='center', ha='left', fontsize=8,
             transform=cbar.ax.get_yaxis_transform())
ax.set_title('Post-Fire Woody Cover Recovery by Ecoregion\n'
             '(Single Fire Events — Pre-Fire Mean = 1.0)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(recovery_plots_dir,
            "eco_recovery_heatmap_prefire_normalized.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_recovery_heatmap_prefire_normalized.png")

# =====================================================
# E10: Recovery Rates Heatmap (1, 3, 5, 10 years)
# =====================================================
print("\nPlot E10: Recovery Rates Heatmap...")

eco_recovery_rows = []
for eco_code, data in eco_results.items():
    if _is_excluded(eco_code): continue
    traj = np.array(data['trajectory'])
    pre_fire = np.nanmean(traj[:5])
    fire_year_val = traj[5]
    loss = pre_fire - fire_year_val
    for years_after in [1, 3, 5, 10]:
        idx = 5 + years_after
        if idx < len(traj):
            rec_val = traj[idx]
            rec_pct = ((rec_val - fire_year_val) / loss * 100) if loss > 0 else np.nan
        else:
            rec_val = np.nan; rec_pct = np.nan
        eco_recovery_rows.append({
            'Ecoregion': eco_code, 'Years_After': years_after,
            'Pre_Fire_Cover': pre_fire, 'Fire_Year_Cover': fire_year_val,
            'Woody_Loss': loss, 'Recovery_Cover': rec_val,
            'Recovery_Percent': rec_pct, 'N_Pixels': data['n_pixels']
        })

df_eco_recovery = pd.DataFrame(eco_recovery_rows)
pivot_rec = df_eco_recovery.pivot_table(
    index='Ecoregion', columns='Years_After', values='Recovery_Percent')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot_rec, cmap='RdYlGn', annot=True, fmt='.0f', ax=ax,
            center=100, linewidths=0.5, cbar_kws={'label': 'Recovery (% of Loss)'})
ax.set_title('Woody Cover Recovery by Ecoregion\n(% of Immediate Loss Recovered)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Years After Fire'); ax.set_ylabel('Ecoregion')
plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "eco_recovery_heatmap.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_recovery_heatmap.png")

# =====================================================
# E11:All Ecoregion Recovery Trajectories (1 Fire) — combined
# =====================================================
print("\nPlot E11: All Ecoregion Recovery Trajectories (1 Fire)...")

fig, ax = plt.subplots(figsize=(14, 8))

for eco_code in sorted(ALL_ECO_CODES):
    data = eco_results.get(eco_code)
    if data is None:
        continue
    
    traj = np.array(data['trajectory'])
    if len(traj) < 16:
        continue
    
    fire_year_cover = traj[5]
    if fire_year_cover <= 0 or np.isnan(fire_year_cover):
        continue
    
    # Normalized recovery trajectory
    traj_post = traj[post_fire_indices] / fire_year_cover
    color = get_eco_color(eco_code)
    eco_name = get_eco_name(eco_code)
    
    ax.plot(
        POST_FIRE_YEARS,
        traj_post,
        marker='o',
        linewidth=2.5,
        label=f"{eco_code} - {eco_name} (n={data['n_pixels']:,})",
        color=color,
        alpha=0.9,
    )

ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.8, alpha=0.7, label='Fire Year (= 1.0)')

ax.set_xlabel('Years After Fire', fontsize=12)
ax.set_ylabel('Normalized Woody Cover', fontsize=12)
ax.set_xlim(-0.3, 10.3)
ax.set_xticks(POST_FIRE_YEARS)
ax.grid(True, alpha=0.3)

ax.set_title(
    "Post-Fire Woody Cover Recovery by Ecoregion\n(Single Fire Events Only — Pre-Fire Mean = 1.0)",
    fontsize=14,
    fontweight="bold",
    pad=20,
)

ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.subplots_adjust(top=0.88)

plt.savefig(
    os.path.join(recovery_plots_dir, "eco_recovery_normalized_all_1fire.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.close()
print("  ✓ eco_recovery_normalized_all_1fire.png")


# --- CSV Exports ---
print("\nSaving Recovery CSVs...")

recovery_norm_df = pd.DataFrame(all_recovery_rows)
recovery_norm_df.to_csv(os.path.join(eco_csv_dir, "eco_recovery_normalized_all_firecounts.csv"), index=False)
print(f"  ✓ eco_recovery_normalized_all_firecounts.csv")

hm_df = pd.DataFrame(hm_matrix, index=[f'+{yr}' for yr in YEARS_AFTER], columns=eco_codes_hm)
hm_df.index.name = 'Year_After_Fire'
hm_df.to_csv(os.path.join(eco_csv_dir, "eco_recovery_heatmap_prefire_normalized.csv"))
print(f"  ✓ eco_recovery_heatmap_prefire_normalized.csv")

df_eco_recovery.to_csv(os.path.join(eco_csv_dir, "recovery_summary_by_ecoregion.csv"), index=False)
print(f"  ✓ recovery_summary_by_ecoregion.csv")

print("\n✓ 4.2c complete")

4.2c  ECOREGION RECOVERY PLOTS

Plot E8: Recovery Normalized Panels...
  ✓ eco_recovery_normalized_1_interannual_fire_panel.png
  ✓ eco_recovery_normalized_2_interannual_fires_panel.png
  ✓ eco_recovery_normalized_3_interannual_fires_panel.png
  ✓ eco_recovery_normalized_4_interannual_fires_panel.png

  Creating combined recovery plot...
  ✓ eco_recovery_normalized_combined_panel.png

Plot E9: Recovery Heatmap (pre-fire normalized)...
  ✓ eco_recovery_heatmap_prefire_normalized.png

Plot E10: Recovery Rates Heatmap...
  ✓ eco_recovery_heatmap.png

Plot E11: All Ecoregion Recovery Trajectories (1 Fire)...
  ✓ eco_recovery_normalized_all_1fire.png

Saving Recovery CSVs...
  ✓ eco_recovery_normalized_all_firecounts.csv
  ✓ eco_recovery_heatmap_prefire_normalized.csv
  ✓ recovery_summary_by_ecoregion.csv

✓ 4.2c complete


In [ ]:
# ============================================================
# 4.2d  ECOREGION — Summary Heatmaps (E11–E12)
# ============================================================
print("=" * 70)
print("4.2d  ECOREGION SUMMARY HEATMAPS")
print("=" * 70)

# =====================================================
# E11: Summary Heatmap (Eco × Fire Count × 4 metrics)
# =====================================================
print("\nPlot E11: Summary Heatmap Eco × Fire Count...")

summary_rows = []
for eco_code, fc_data in eco_firecount_results.items():
    if _is_excluded(eco_code): continue
    eco_name = get_eco_name(eco_code)
    for fc, data in fc_data.items():
        traj = np.array(data['trajectory'])
        pre_fire = np.nanmean(traj[:5])
        fire_year_val = traj[5]
        loss = pre_fire - fire_year_val
        rec_5yr = traj[10] if len(traj) > 10 else np.nan
        rec_5yr_pct = ((rec_5yr - fire_year_val) / loss * 100) if loss > 0 else np.nan
        rec_10yr = traj[15] if len(traj) > 15 else np.nan
        rec_10yr_pct = ((rec_10yr - fire_year_val) / loss * 100) if (loss > 0 and not np.isnan(rec_10yr)) else np.nan
        summary_rows.append({
            'Ecoregion': eco_code, 'Ecoregion_Name': eco_name,
            'Fire_Count': fc, 'N_Pixels': data['n_pixels'],
            'Area_km2': data['n_pixels'] * pixel_area_km2,
            'Woody_Loss': loss,
            'Recovery_5yr_Pct': rec_5yr_pct, 'Recovery_10yr_Pct': rec_10yr_pct
        })

df_summary = pd.DataFrame(summary_rows)

metrics_hm = ['Area_km2', 'Woody_Loss', 'Recovery_5yr_Pct', 'Recovery_10yr_Pct']
titles_hm = ['Affected Area (km²)', 'Immediate Loss (%)',
             '5-Year Recovery (%)', '10-Year Recovery (%)']
cmaps_hm = ['YlOrRd', 'Reds', 'RdYlGn', 'RdYlGn']
centers_hm = [None, None, 100, 100]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes_flat = axes.flatten()
for idx, (metric, title, cmap, center) in enumerate(zip(metrics_hm, titles_hm, cmaps_hm, centers_hm)):
    ax = axes_flat[idx]
    pivot = df_summary.pivot_table(index='Fire_Count', columns='Ecoregion', values=metric)
    sns.heatmap(pivot, cmap=cmap, annot=True,
                fmt='.1f' if 'Area' in metric else '.0f',
                ax=ax, linewidths=0.5, center=center, cbar_kws={'label': title})
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Fire Count')
    ax.set_yticklabels([f'{int(fc)} Fire(s)' for fc in pivot.index])

plt.suptitle('Comprehensive Summary: Ecoregion × Fire Count',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "eco_summary_heatmap.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_summary_heatmap.png")

# =====================================================
# E12: Summary Heatmap Ecoregion × Fire Count (Area/Loss/Recovery)
# =====================================================
print("\nPlot E12: Ecoregion × Fire Count (detailed)...")

eco_total_areas_km2 = {
    eco_mapping[eco_id]['code']: float(np.sum(eco_raster == eco_id)) * pixel_area_km2
    for eco_id in unique_eco_ids
}

summary_stats_fc = []
for eco_id in unique_eco_ids:
    eco_code = eco_mapping[eco_id]['code']
    eco_name = eco_mapping[eco_id]['name']
    if eco_code not in eco_firecount_results: continue
    eco_total_area = eco_total_areas_km2[eco_code]
    for fc_val, traj_data in eco_firecount_results[eco_code].items():
        traj_arr = np.array(traj_data['trajectory'])
        pre_fire = float(np.nanmean(traj_arr[:5]))
        fire_year = float(traj_arr[5])
        rec_5yr = float(traj_arr[10])
        loss = pre_fire - fire_year
        rec_pct_5yr = ((rec_5yr - fire_year) / loss * 100) if loss > 0 else np.nan
        area_km2 = traj_data['n_pixels'] * pixel_area_km2
        summary_stats_fc.append({
            'Ecoregion_Code': eco_code, 'Ecoregion_Name': eco_name,
            'Fire_Count': fc_val, 'N_Pixels': traj_data['n_pixels'],
            'Area_km2': area_km2,
            'Percent_of_Ecoregion': (area_km2 / eco_total_area * 100) if eco_total_area > 0 else 0,
            'Woody_Loss': loss, 'Recovery_Percent_5yr': rec_pct_5yr
        })

summary_df_eco = pd.DataFrame(summary_stats_fc)

metrics_fc  = ['Area_km2', 'Percent_of_Ecoregion', 'Woody_Loss', 'Recovery_Percent_5yr']
titles_fc   = ['Affected Area (km²)', 'Affected Area (% of Region)',
               'Woody Cover Loss (%)', 'Recovery after 5 Years (%)']
cmaps_fc    = ['YlOrRd', 'YlOrRd', 'Reds', 'RdYlGn']

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
for idx, (metric, title, cmap) in enumerate(zip(metrics_fc, titles_fc, cmaps_fc)):
    ax = axes[idx // 2, idx % 2]
    pivot = summary_df_eco.pivot(index='Fire_Count', columns='Ecoregion_Code', values=metric)
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap)
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticklabels([f'{int(fc)} Fire(s)' for fc in pivot.index])
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                tc = 'white' if val > np.nanmean(pivot.values) else 'black'
                txt = f'{val:.1f}' if metric == 'Area_km2' else f'{val:.1f}%'
                ax.text(j, i, txt, ha='center', va='center',
                        color=tc, fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Ecoregion', fontsize=11); ax.set_ylabel('Fire Count', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(labelsize=9)

plt.suptitle('Summary Heatmap: Ecoregion × Fire Count',
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(eco_plots_dir, "summary_heatmap_ecoregion_firecount.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ summary_heatmap_ecoregion_firecount.png")

# --- CSV Export ---
print("\nSaving Summary CSV...")
summary_df_eco.to_csv(os.path.join(eco_csv_dir, "summary_statistics_by_fire_count.csv"), index=False)
print(f"  ✓ summary_statistics_by_fire_count.csv")

print(f"\n{'='*70}")
print(f"✓ 4.2 ECOREGION VISUALIZATIONS COMPLETE")
print(f"  Plots: {eco_plots_dir}")
print(f"  Recovery: {recovery_plots_dir}")
print(f"  CSVs: {eco_csv_dir}")
print(f"{'='*70}")

In [ ]:
# ============================================================
# 4.3  LANDCOVER VISUALIZATIONS
# ============================================================
print("=" * 70)
print("4.3  LANDCOVER VISUALIZATIONS")
print("=" * 70)

# =====================================================
# L1: All LC Trajectories (1 Fire) — combined
# =====================================================
print("\nPlot L1: All LC Trajectories (1 Fire)...")

fig, ax = plt.subplots(figsize=(14, 8))

for lc_class, data in sorted(lc_major_results.items()):
    color = LC_COLORS.get(lc_class, '#808080')
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])

    ax.plot(rel_years, traj, marker='o', linewidth=2.5,
            label=f"{lc_class} (n={data['n_pixels']:,})",
            color=color, alpha=0.9)

    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)
    ax.fill_between(rel_years, lower, upper, alpha=0.15, color=color)

ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Fire Year')
ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('Years Relative to Fire Event', fontsize=12)
ax.set_ylabel('Woody Cover (%)', fontsize=12)
ax.set_title('Post-Fire Woody Cover Trajectories by Pre-Fire Land Cover Type\n'
             '(Single Fire Events Only)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5.5, 10.5)

ax.annotate('Pre-Fire', xy=(-3, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkgreen', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
ax.annotate('Fire\nYear', xy=(0.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkred', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.5))
ax.annotate('Recovery Period', xy=(5.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkblue', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.tight_layout()
plt.savefig(os.path.join(lc_plots_dir, "lc_trajectories_1fire_all.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ lc_trajectories_1fire_all.png")

# =====================================================
# L2: Subplots per LC class (1 Fire)
# =====================================================
print("\nPlot L2: Subplots per LC class...")

n_lc = len(lc_major_results)
nc_lc, nr_lc = 3, int(np.ceil(n_lc / 3))

fig, axes = plt.subplots(nr_lc, nc_lc, figsize=(16, 5 * nr_lc))
axes_flat = axes.flatten() if n_lc > 1 else [axes]

for i, (lc_class, data) in enumerate(sorted(lc_major_results.items())):
    ax = axes_flat[i]
    color = LC_COLORS.get(lc_class, '#808080')
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])
    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)

    ax.plot(rel_years, traj, marker='o', linewidth=2.5, color=color)
    ax.fill_between(rel_years, lower, upper, alpha=0.3, color=color)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.set_ylim(max(0, np.nanmin(lower) - 5), min(100, np.nanmax(upper) + 5))
    ax.set_title(f"{lc_class}\n(n={data['n_pixels']:,} Events)", fontsize=11, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=9)
    ax.set_ylabel('Woody Cover (%)', fontsize=9)
    ax.grid(True, alpha=0.3)
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)

for i in range(n_lc, len(axes_flat)):
    axes_flat[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Pre-Fire Land Cover (Single Fire Events)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(lc_plots_dir, "lc_trajectories_1fire_panel.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ lc_trajectories_1fire_panel.png")

# =====================================================
# L3: LC × Fire Count
# =====================================================
print("\nPlot L3: LC × Fire Count...")

lc_with_fc = {lc: d for lc, d in lc_firecount_results.items() if d}
n_lc_fc = len(lc_with_fc)
nc3, nr3 = 2, int(np.ceil(n_lc_fc / 2))

fig, axes = plt.subplots(nr3, nc3, figsize=(16, 6 * nr3))
axes_flat = axes.flatten() if n_lc_fc > 1 else [axes]

for plot_idx, lc_class in enumerate(sorted(lc_with_fc.keys())):
    fc_data = lc_with_fc[lc_class]
    ax = axes_flat[plot_idx]

    for fc, data in sorted(fc_data.items()):
        traj = np.array(data['trajectory'])
        std = np.array(data['std'])
        color = FIRE_COUNT_COLORS.get(fc, '#808080')
        ax.plot(rel_years, traj, marker='o', linewidth=2.5, color=color,
                label=f'{fc} Fire(s) (n={data["n_pixels"]:,})', alpha=0.8)
        lower = np.maximum(traj - std, 0)
        upper = np.minimum(traj + std, 100)
        ax.fill_between(rel_years, lower, upper, alpha=0.15, color=color)

    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.set_title(f"{lc_class}", fontsize=12, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=10)
    ax.set_ylabel('Woody Cover (%)', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)
    lc_color = LC_COLORS.get(lc_class, '#808080')
    for spine in ax.spines.values():
        spine.set_edgecolor(lc_color)
        spine.set_linewidth(2)

for i in range(n_lc_fc, len(axes_flat)):
    axes_flat[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Land Cover × Fire Count',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(lc_plots_dir, "lc_trajectories_by_fire_count.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ lc_trajectories_by_fire_count.png")

# --- CSV Exports ---
print("\nSaving LC CSVs...")

rows = []
for lc_class, data in sorted(lc_major_results.items()):
    for i, ry in enumerate(rel_years):
        rows.append({'Landcover': lc_class, 'N_Pixels': data['n_pixels'],
                     'Rel_Year': ry,
                     'Woody_Cover_Mean': data['trajectory'][i],
                     'Woody_Cover_Std': data['std'][i]})
pd.DataFrame(rows).to_csv(os.path.join(lc_csv_dir, "trajectories_by_landcover_1fire.csv"), index=False)
print(f"  ✓ trajectories_by_landcover_1fire.csv")

rows = []
for lc_class, fc_data in sorted(lc_firecount_results.items()):
    if not fc_data: continue
    for fc, data in sorted(fc_data.items()):
        for i, ry in enumerate(rel_years):
            rows.append({'Landcover': lc_class, 'Fire_Count': fc,
                         'N_Pixels': data['n_pixels'], 'Rel_Year': ry,
                         'Woody_Cover_Mean': data['trajectory'][i],
                         'Woody_Cover_Std': data['std'][i]})
pd.DataFrame(rows).to_csv(os.path.join(lc_csv_dir, "trajectories_by_landcover_x_firecount.csv"), index=False)
print(f"  ✓ trajectories_by_landcover_x_firecount.csv")

# Recovery summary by LC
lc_recovery_rows = []
for lc_class, data in sorted(lc_major_results.items()):
    traj = np.array(data['trajectory'])
    pre_fire = np.nanmean(traj[:5])
    fire_year_val = traj[5]
    loss = pre_fire - fire_year_val
    for years_after in [1, 3, 5, 10]:
        idx = 5 + years_after
        if idx < len(traj):
            rec_val = traj[idx]
            rec_pct = ((rec_val - fire_year_val) / loss * 100) if loss > 0 else np.nan
        else:
            rec_val = np.nan; rec_pct = np.nan
        lc_recovery_rows.append({
            'Landcover': lc_class, 'Years_After': years_after,
            'Pre_Fire_Cover': pre_fire, 'Fire_Year_Cover': fire_year_val,
            'Woody_Loss': loss, 'Recovery_Cover': rec_val,
            'Recovery_Percent': rec_pct, 'N_Pixels': data['n_pixels']
        })
pd.DataFrame(lc_recovery_rows).to_csv(os.path.join(lc_csv_dir, "recovery_summary_by_landcover.csv"), index=False)
print(f"  ✓ recovery_summary_by_landcover.csv")

print(f"\n{'='*70}")
print(f"✓ 4.3 LANDCOVER VISUALIZATIONS COMPLETE")
print(f"  Plots: {lc_plots_dir}")
print(f"  CSVs: {lc_csv_dir}")
print(f"{'='*70}")

In [40]:
# ============================================================
# 4.4  COMBINED: LANDCOVER × ECOREGION
# ============================================================
print("=" * 70)
print("4.4  COMBINED: LANDCOVER × ECOREGION")
print("=" * 70)

# =====================================================
# C1: Heatmap LC × Ecoregion (Loss / Recovery / Sample Size)
# =====================================================
print("\nPlot C1: Heatmap LC × Ecoregion...")

heatmap_data_loss = {}
heatmap_data_recovery = {}
heatmap_data_npixels = {}

for eco_code, lc_data in lc_eco_results.items():
    heatmap_data_loss[eco_code] = {}
    heatmap_data_recovery[eco_code] = {}
    heatmap_data_npixels[eco_code] = {}

    for lc_class, data in lc_data.items():
        traj = np.array(data['trajectory'])
        pre_fire = np.nanmean(traj[:5])
        fire_year = traj[5]
        recovery_5yr = traj[10] if len(traj) > 10 else traj[-1]
        loss = pre_fire - fire_year if not np.isnan(fire_year) else np.nan
        rec_pct = ((recovery_5yr - fire_year) / loss * 100) if loss > 0 else np.nan

        heatmap_data_loss[eco_code][lc_class] = loss
        heatmap_data_recovery[eco_code][lc_class] = rec_pct
        heatmap_data_npixels[eco_code][lc_class] = data['n_pixels']

df_loss = pd.DataFrame(heatmap_data_loss).T
df_recovery_hm = pd.DataFrame(heatmap_data_recovery).T
df_npixels = pd.DataFrame(heatmap_data_npixels).T

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

ax = axes[0]
sns.heatmap(df_loss, cmap='Reds', annot=True, fmt='.1f', ax=ax,
            cbar_kws={'label': 'Woody Cover Loss (%)'})
ax.set_title('Immediate Woody Cover Loss\n(Pre-Fire − Fire Year)', fontsize=12, fontweight='bold')
ax.set_xlabel('Land Cover Type'); ax.set_ylabel('Ecoregion')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

ax = axes[1]
sns.heatmap(df_recovery_hm, cmap='RdYlGn', annot=True, fmt='.0f', ax=ax,
            center=100, cbar_kws={'label': 'Recovery after 5 Years (%)'})
ax.set_title('Recovery after 5 Years (%)\n(100% = Full Recovery)', fontsize=12, fontweight='bold')
ax.set_xlabel('Land Cover Type'); ax.set_ylabel('Ecoregion')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

ax = axes[2]
sns.heatmap(df_npixels, cmap='Blues', annot=True, fmt='.0f', ax=ax,
            cbar_kws={'label': 'Number of Pixels'})
ax.set_title('Sample Size\n(Pixels with Single Fire)', fontsize=12, fontweight='bold')
ax.set_xlabel('Land Cover Type'); ax.set_ylabel('Ecoregion')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Land Cover × Ecoregion: Fire Impact & Recovery Summary',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(lcxeco_plots_dir, "lc_ecoregion_heatmap.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ lc_ecoregion_heatmap.png")

# # =====================================================
# # C2: LC Trajectories per Ecoregion (one plot per region)
# # =====================================================
# print("\nPlot C2: LC Trajectories per Ecoregion...")

# for eco_code, lc_data in sorted(lc_eco_results.items()):
#     if not lc_data:
#         continue

#     fig, ax = plt.subplots(figsize=(14, 8))

#     for lc_class, data in sorted(lc_data.items()):
#         color = LC_COLORS.get(lc_class, '#808080')
#         traj = np.array(data['trajectory'])
#         ax.plot(rel_years, traj, marker='o', linewidth=2.5,
#                 label=f"{lc_class} (n={data['n_pixels']:,})",
#                 color=color, alpha=0.9)

#     ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
#     ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)

#     eco_name = eco_mapping.get(
#         [k for k, v in eco_mapping.items() if v['code'] == eco_code][0], {}
#     ).get('name', eco_code)

#     ax.set_title(f'{eco_code} – {eco_name}\nWoody Cover Trajectories by Pre-Fire Land Cover',
#                  fontsize=13, fontweight='bold')
#     ax.set_xlabel('Years Relative to Fire Event', fontsize=12)
#     ax.set_ylabel('Woody Cover (%)', fontsize=12)
#     ax.legend(loc='best', fontsize=10)
#     ax.grid(True, alpha=0.3)
#     ax.set_xlim(-5.5, 10.5)

#     plt.tight_layout()
#     plt.savefig(os.path.join(lcxeco_plots_dir, f"lc_trajectories_{eco_code}.png"),
#                 dpi=300, bbox_inches='tight')
#     plt.close()
#     print(f"  ✓ lc_trajectories_{eco_code}.png")

# =====================================================
# C2: LC Trajectories per Ecoregion (Panel)
# =====================================================
print("\nPlot C2: LC Trajectories per Ecoregion (Panel)...")

n_eco_c2 = len(ALL_ECO_CODES)
n_cols_c2 = 3
n_rows_c2 = int(np.ceil(n_eco_c2 / n_cols_c2))

fig, axes = plt.subplots(n_rows_c2, n_cols_c2, figsize=(16, 5 * n_rows_c2))
axes_flat = axes.flatten()

# Collect legend handles from first ecoregion (all LC classes have same color)
_legend_handles = []
_legend_labels = []

for idx, eco_code in enumerate(sorted(ALL_ECO_CODES)):
    ax = axes_flat[idx]
    lc_data = lc_eco_results.get(eco_code, {})
    
    if not lc_data:
        ax.axis('off')
        continue
    
    has_plot = False
    for lc_class, data in sorted(lc_data.items()):
        color = LC_COLORS.get(lc_class, '#808080')
        traj = np.array(data['trajectory'])
        line, = ax.plot(rel_years, traj, marker='o', linewidth=2.5,
                        label=f"{lc_class} (n={data['n_pixels']:,})",
                        color=color, alpha=0.9)
        if idx == 0:
            _legend_handles.append(line)
            _legend_labels.append(f"{lc_class} (n={data['n_pixels']:,})")
        has_plot = True
    
    if not has_plot:
        ax.axis('off')
        continue
    
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    eco_name = get_eco_name(eco_code)
    ax.set_title(f'{eco_code} – {eco_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire Event', fontsize=10)
    ax.set_ylabel('Woody Cover (%)', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)

# Single legend at top right (fixed position)
if _legend_handles:
    fig.legend(_legend_handles, _legend_labels, loc='upper right', 
               fontsize=10, bbox_to_anchor=(0.98, 0.98))

# Disable unused subplots
for idx in range(n_eco_c2, len(axes_flat)):
    axes_flat[idx].axis('off')

plt.suptitle('Woody Cover Trajectories by Pre-Fire Land Cover (LC × Ecoregion Panel)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(top=0.92, right=0.85)
plt.savefig(os.path.join(lcxeco_plots_dir, "lc_trajectories_panel.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ lc_trajectories_panel.png")

# =====================================================
# C3: Summary Heatmap Ecoregion × LC
# =====================================================
print("\nPlot C3: Summary Heatmap Ecoregion × LC...")

summary_lcxeco_rows = []
for eco_code, lc_data in lc_eco_results.items():
    if _is_excluded(eco_code): continue
    eco_name = get_eco_name(eco_code)
    for lc_class, data in lc_data.items():
        traj = np.array(data['trajectory'])
        pre_fire = np.nanmean(traj[:5])
        fire_year_val = traj[5]
        loss = pre_fire - fire_year_val
        rec_5yr = traj[10] if len(traj) > 10 else np.nan
        rec_5yr_pct = ((rec_5yr - fire_year_val) / loss * 100) if loss > 0 else np.nan
        summary_lcxeco_rows.append({
            'Ecoregion': eco_code, 'Ecoregion_Name': eco_name,
            'Landcover': lc_class, 'N_Pixels': data['n_pixels'],
            'Area_km2': data['n_pixels'] * pixel_area_km2,
            'Pre_Fire_Cover': pre_fire, 'Fire_Year_Cover': fire_year_val,
            'Woody_Loss': loss, 'Recovery_5yr_Pct': rec_5yr_pct
        })

df_lcxeco = pd.DataFrame(summary_lcxeco_rows)

if len(df_lcxeco) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))

    pivot_loss = df_lcxeco.pivot_table(index='Ecoregion', columns='Landcover', values='Woody_Loss')
    sns.heatmap(pivot_loss, cmap='Reds', annot=True, fmt='.1f', ax=axes[0],
                linewidths=0.5, cbar_kws={'label': 'Woody Cover Loss (%)'})
    axes[0].set_title('Immediate Woody Cover Loss (%)', fontsize=12, fontweight='bold')

    pivot_rec = df_lcxeco.pivot_table(index='Ecoregion', columns='Landcover', values='Recovery_5yr_Pct')
    sns.heatmap(pivot_rec, cmap='RdYlGn', annot=True, fmt='.0f', ax=axes[1],
                center=100, linewidths=0.5, cbar_kws={'label': '5-Year Recovery (%)'})
    axes[1].set_title('5-Year Recovery (% of Loss)', fontsize=12, fontweight='bold')

    pivot_area = df_lcxeco.pivot_table(index='Ecoregion', columns='Landcover', values='Area_km2')
    sns.heatmap(pivot_area, cmap='YlOrRd', annot=True, fmt='.0f', ax=axes[2],
                linewidths=0.5, cbar_kws={'label': 'Area (km²)'})
    axes[2].set_title('Affected Area (km²)', fontsize=12, fontweight='bold')

    for ax in axes:
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    plt.suptitle('Summary: Ecoregion × Land Cover',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(lcxeco_plots_dir, "summary_heatmap_ecoregion_lc.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ summary_heatmap_ecoregion_lc.png")

    # =====================================================
    # C4: Representative Pixel Trajectories (1–4 Fire Events)
    # =====================================================
    print("\nPlot C4: Representative Pixel Trajectories by Fire Count...")

    pixel_traj_dir = os.path.join(lcxeco_plots_dir, "pixel_trajectories")
    os.makedirs(pixel_traj_dir, exist_ok=True)

    # --- Ensure raw rasters are available (not in cache) ---
    if not isinstance(globals().get('burned'), np.ndarray):
        print("  Loading burned raster...")
        with rasterio.open(burned_path) as src:
            burned = src.read([1 + (yr - MBA_START_YEAR) * len(band_structure)
                               for yr in years_burned])
        print(f"    ✓ burned: {burned.shape}")

    if not isinstance(globals().get('woody'), np.ndarray):
        print("  Loading woody raster...")
        with rasterio.open(woody_path) as src:
            woody = src.read()
        print(f"    ✓ woody: {woody.shape}")

    with rasterio.open(woody_path) as src:
        _nodata_val = src.nodata

    # fire_counts as ndarray (may be list from cache)
    if isinstance(globals().get('fire_counts'), np.ndarray):
        _fc_arr = fire_counts
    else:
        _fc_arr = np.sum(burned == 1, axis=0)

    if not isinstance(globals().get('first_fire_idx'), np.ndarray):
        first_fire_idx = np.argmax(burned == 1, axis=0)

    if not isinstance(globals().get('pre_fire_lc_major_id'), np.ndarray):
        print("  Computing pre-fire landcover...")
        if not isinstance(globals().get('modis_igbp'), np.ndarray):
            with rasterio.open(modis_path) as src:
                modis_igbp = src.read()
        _igbp_map = np.array([0,1,1,1,1,1,2,2,3,3,4,5,6,7,6,8,8,9], dtype=np.uint8)
        _hf = _fc_arr > 0
        _fy = np.array(years_burned)[first_fire_idx]
        _mi = np.clip(_fy - 1 - modis_years[0], 0, len(modis_years) - 1)
        _h, _w = eco_raster.shape
        _plc = np.zeros((_h, _w), dtype=np.uint8)
        _r, _c = np.where(_hf)
        _plc[_r, _c] = modis_igbp[_mi[_r, _c], _r, _c]
        _plc[(_plc < 1) | (_plc > 17)] = 0
        pre_fire_lc_major_id = _igbp_map[_plc]
        pre_fire_lc_major_id[pre_fire_lc_major_id == 8] = 0
        print(f"    ✓ pre_fire_lc_major_id computed")

    # --- Constants ---
    _LC_TO_ID = {'Forests': 1, 'Shrublands': 2, 'Open Woodland': 3,
                 'Grasslands': 4, 'Wetlands': 5, 'Croplands': 6}
    _ECO_TO_ID = {v['code']: k for k, v in eco_mapping.items()}
    _WD_OFFSET = years_burned[0] - years_woody[0]   # 16
    _PLOT_YEARS = list(range(2001, 2025))             # 24 years
    _N_PY = len(_PLOT_YEARS)

    # --- Find representative pixel (median woody cover at first fire) ---
    def _find_rep_pixel(eco_id, lc_id, fc):
        mask = ((eco_raster == eco_id) &
                (pre_fire_lc_major_id == lc_id) &
                (_fc_arr == fc))
        rs, cs = np.where(mask)
        if len(rs) < 5:
            return None
        wi = first_fire_idx[rs, cs] + _WD_OFFSET
        ok = (wi >= 0) & (wi < len(years_woody))
        if ok.sum() < 5:
            return None
        rs, cs, wi = rs[ok], cs[ok], wi[ok]
        vals = woody[wi, rs, cs].astype(float)
        if _nodata_val is not None:
            good = np.isfinite(vals) & (vals != _nodata_val)
        else:
            good = np.isfinite(vals)
        if good.sum() < 5:
            return None
        rs, cs, vals = rs[good], cs[good], vals[good]
        best = np.argmin(np.abs(vals - np.median(vals)))
        return int(rs[best]), int(cs[best])

    # --- Build representative pixel dict ---
    print("  Selecting representative pixels...")
    _lc_classes = sorted(LC_COLORS.keys())
    rep_pixels = {}
    pixel_csv_rows = []

    for fc in [1, 2, 3, 4]:
        n_found = 0
        for eco_code in ALL_ECO_CODES:
            eid = _ECO_TO_ID.get(eco_code)
            if eid is None:
                continue
            for lc in _lc_classes:
                lid = _LC_TO_ID.get(lc)
                if lid is None:
                    continue
                px = _find_rep_pixel(eid, lid, fc)
                if px is None:
                    continue
                r, c = px
                fi = np.where(burned[:, r, c] == 1)[0]
                fy_list = [years_burned[i] for i in fi]
                wc = woody[_WD_OFFSET:_WD_OFFSET + _N_PY, r, c].astype(float)
                if _nodata_val is not None:
                    wc[wc == _nodata_val] = np.nan
                rep_pixels[(eco_code, lc, fc)] = {
                    'row': r, 'col': c, 'fire_years': fy_list, 'wc': wc}
                pixel_csv_rows.append({
                    'Ecoregion': eco_code, 'Landcover': lc, 'Fire_Count': fc,
                    'Row': r, 'Col': c,
                    'Fire_Years': ';'.join(map(str, fy_list))})
                n_found += 1
        print(f"    FC={fc}: {n_found}/{len(ALL_ECO_CODES)*len(_lc_classes)} pixels found")

    print(f"  ✓ {len(rep_pixels)} representative pixels selected")

    # --- Plot: one figure per fire count ---
    _n_eco = len(ALL_ECO_CODES)
    _n_lc = len(_lc_classes)

    for fc in [1, 2, 3, 4]:
        fig, axes = plt.subplots(
            _n_eco, _n_lc,
            figsize=(3.5 * _n_lc, 2.5 * _n_eco),
            squeeze=False)

        for i, eco_code in enumerate(ALL_ECO_CODES):
            eco_col = get_eco_color(eco_code)
            for j, lc in enumerate(_lc_classes):
                ax = axes[i, j]
                pxd = rep_pixels.get((eco_code, lc, fc))

                if pxd is None:
                    ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                            fontsize=9, color='lightgray', transform=ax.transAxes)
                else:
                    ax.plot(_PLOT_YEARS, pxd['wc'], marker='o', ms=3, lw=1.5,
                            color=LC_COLORS.get(lc, '#808080'), alpha=0.9)
                    for fy in pxd['fire_years']:
                        if 2001 <= fy <= 2024:
                            ax.axvline(x=fy, color='red', ls='--', lw=1.5, alpha=0.7)

                ax.set_xlim(2000, 2025)
                ax.set_ylim(0, 100)
                for sp in ax.spines.values():
                    sp.set_edgecolor(eco_col)
                    sp.set_linewidth(1.5)
                ax.grid(True, alpha=0.2)
                ax.tick_params(labelsize=7)

                if i == 0:
                    ax.set_title(lc, fontsize=10, fontweight='bold')
                if j == 0:
                    ax.set_ylabel(f'{eco_code}\nWoody (%)', fontsize=8)
                else:
                    ax.set_ylabel('')
                if i == _n_eco - 1:
                    ax.set_xticks([2005, 2010, 2015, 2020])
                    ax.set_xticklabels(['05', '10', '15', '20'], fontsize=7)
                else:
                    ax.set_xticklabels([])

        fc_str = f"{fc} Fire Event{'s' if fc > 1 else ''}"
        fig.suptitle(
            f'Representative Pixel Trajectories — {fc_str}\n'
            f'Woody Cover (2001–2024) · Red lines = Fire Year',
            fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.subplots_adjust(top=0.94)
        out_name = f"pixel_trajectories_{fc}fires.png"
        plt.savefig(os.path.join(pixel_traj_dir, out_name),
                    dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  ✓ {out_name}")

    # --- CSV: representative pixel coordinates & fire years ---
    if pixel_csv_rows:
        pd.DataFrame(pixel_csv_rows).to_csv(
            os.path.join(lcxeco_csv_dir, "representative_pixels.csv"), index=False)
        print(f"  ✓ representative_pixels.csv")

    print(f"\n  ✓ C4 complete — {len(rep_pixels)} pixel trajectories")    

# --- CSV Exports ---
print("\nSaving Combined CSVs...")

rows = []
for eco_code, lc_data in sorted(lc_eco_results.items()):
    for lc_class, data in sorted(lc_data.items()):
        for i, ry in enumerate(rel_years):
            rows.append({'Ecoregion': eco_code, 'Landcover': lc_class,
                         'N_Pixels': data['n_pixels'], 'Rel_Year': ry,
                         'Woody_Cover_Mean': data['trajectory'][i],
                         'Woody_Cover_Std': data['std'][i]})
pd.DataFrame(rows).to_csv(os.path.join(lcxeco_csv_dir, "trajectories_by_ecoregion_x_landcover.csv"), index=False)
print(f"  ✓ trajectories_by_ecoregion_x_landcover.csv")

if len(df_lcxeco) > 0:
    df_lcxeco.to_csv(os.path.join(lcxeco_csv_dir, "summary_ecoregion_x_landcover.csv"), index=False)
    print(f"  ✓ summary_ecoregion_x_landcover.csv")

print(f"\n{'='*70}")
print(f"✓ 4.4 COMBINED VISUALIZATIONS COMPLETE")
print(f"  Plots: {lcxeco_plots_dir}")
print(f"  CSVs: {lcxeco_csv_dir}")
print(f"{'='*70}")

4.4  COMBINED: LANDCOVER × ECOREGION

Plot C1: Heatmap LC × Ecoregion...
  ✓ lc_ecoregion_heatmap.png

Plot C2: LC Trajectories per Ecoregion (Panel)...
  ✓ lc_trajectories_panel.png

Plot C3: Summary Heatmap Ecoregion × LC...
  ✓ summary_heatmap_ecoregion_lc.png

Plot C4: Representative Pixel Trajectories by Fire Count...
  Loading burned raster...
    ✓ burned: (25, 9660, 10483)
  Loading woody raster...
    ✓ woody: (40, 9660, 10483)
  Computing pre-fire landcover...
    ✓ pre_fire_lc_major_id computed
  Selecting representative pixels...
    FC=1: 56/60 pixels found
    FC=2: 45/60 pixels found
    FC=3: 40/60 pixels found
    FC=4: 36/60 pixels found
  ✓ 177 representative pixels selected
  ✓ pixel_trajectories_1fires.png
  ✓ pixel_trajectories_2fires.png
  ✓ pixel_trajectories_3fires.png
  ✓ pixel_trajectories_4fires.png
  ✓ representative_pixels.csv

  ✓ C4 complete — 177 pixel trajectories

Saving Combined CSVs...
  ✓ trajectories_by_ecoregion_x_landcover.csv
  ✓ summary_ecoreg

In [4]:
# =====================================================
# C2: LC Trajectories per Ecoregion (Panel)
# =====================================================
print("\nPlot C2: LC Trajectories per Ecoregion (Panel)...")

n_eco_c2 = len(ALL_ECO_CODES)
n_cols_c2 = 3
n_rows_c2 = int(np.ceil(n_eco_c2 / n_cols_c2))

fig, axes = plt.subplots(n_rows_c2, n_cols_c2, figsize=(16, 5 * n_rows_c2))
axes_flat = axes.flatten()

for idx, eco_code in enumerate(sorted(ALL_ECO_CODES)):
    ax = axes_flat[idx]
    lc_data = lc_eco_results.get(eco_code, {})
    
    if not lc_data:
        ax.axis('off')
        continue
    
    has_plot = False
    for lc_class, data in sorted(lc_data.items()):
        color = LC_COLORS.get(lc_class, '#808080')
        traj = np.array(data['trajectory'])
        ax.plot(rel_years, traj, marker='o', linewidth=2.5,
                label=f"{lc_class} (n={data['n_pixels']:,})",
                color=color, alpha=0.9)
        has_plot = True
    
    if not has_plot:
        ax.axis('off')
        continue
    
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    eco_name = get_eco_name(eco_code)
    ax.set_title(f'{eco_code}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire Event', fontsize=10)
    ax.set_ylabel('Woody Cover (%)', fontsize=10)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)

# Disable unused subplots
for idx in range(n_eco_c2, len(axes_flat)):
    axes_flat[idx].axis('off')

# plt.suptitle('Woody Cover Trajectories by Pre-Fire Land Cover (LC × Ecoregion Panel)',
#              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(lcxeco_plots_dir, "lc_trajectories_eco_panel.png"),
            dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ lc_trajectories_eco_panel.png")


Plot C2: LC Trajectories per Ecoregion (Panel)...
  ✓ lc_trajectories_eco_panel.png
